<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline/mnps_new_baseline%20v6.0.2.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 6.0.2.2**
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 24, 2025  
> Drafted by Wayne Birch - [contact him](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).





## **2** | Environment Setup
We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.


> # **Version 6.0 Change**
> Added second API call to determine confidence level of each record against each roll.


In [1]:
#Cell 3
!pip install -U openai

In [2]:
#Cell 3.5
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): GPT-4o
✅ Using MODEL_ID: gpt-4o-2024-11-20


In [3]:
# ===== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Fixed output location (as requested) ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Where to find your three inputs by default ----------
# If you want to upload instead of copying from Drive, set ALLOW_UPLOAD = True.
DATA_INPUTS_DIR = Path("/content/drive/My Drive/Colab Notebooks/Data Inputs")
ALLOW_UPLOAD = False  # set True to be prompted to upload the 3 files from your computer

REQUIRED = {
    "Ground Truth Masterfile.csv": DATA_INPUTS_DIR / "Ground Truth Masterfile.csv",
    "Sample JDs.csv":  DATA_INPUTS_DIR / "Sample JDs.csv",
    "MNPS_Prompt_Resources.zip":  DATA_INPUTS_DIR / "MNPS_Prompt_Resources.zip",
}

# (A) Optionally upload files instead of copying from Drive
if ALLOW_UPLOAD:
    from google.colab import files as colab_files
    print("🔼 Upload the three files when prompted:")
    uploaded = colab_files.upload()  # opens a browser picker
    for name in REQUIRED.keys():
        if name in uploaded:
            dst = INPUTS_DIR / name
            with open(dst, "wb") as f:
                f.write(uploaded[name])
            REQUIRED[name] = dst  # point to the just-uploaded copy

# (B) Copy from Drive if not already present in /inputs
missing = []
for name, src in REQUIRED.items():
    dst = INPUTS_DIR / name
    if dst.exists():
        continue
    if src.exists():
        shutil.copy2(src, dst)
        print(f"📄 Copied: {src}  →  {dst}")
    else:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "These input files were not found. Place them in "
        f"{DATA_INPUTS_DIR} or enable ALLOW_UPLOAD=True:\n - " + "\n - ".join(missing)
    )

# ---------- 3) Unpack the resources zip into inputs/resources (optional but helpful) ----------
resources_zip = INPUTS_DIR / "MNPS_Prompt_Resources.zip"
RESOURCES_DIR = INPUTS_DIR / "resources"
if resources_zip.exists():
    RESOURCES_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(resources_zip, "r") as zf:
        zf.extractall(RESOURCES_DIR)
    print("🧰 Unpacked resources to:", RESOURCES_DIR)

# ---------- 4) Robust CSV reader (handles cp1252/latin1) ----------
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    trials = [
        dict(encoding="utf-8"),
        dict(encoding="utf-8-sig"),
        dict(encoding="cp1252"),
        dict(encoding="latin1"),
    ]
    for t in trials:
        try:
            df = pd.read_csv(path, **{**t, **kwargs})
            print(f"✅ Read {path.name} with encoding={t['encoding']}")
            return df
        except UnicodeDecodeError:
            continue
    # last resort
    df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip", **kwargs)
    print(f"⚠️ Read {path.name} with encoding=latin1 (on_bad_lines='skip')")
    return df

# Smoke test: load one row from the sample CSV (row 0) and build job_desc_text for downstream cells
sample_csv = INPUTS_DIR / "Sample JDs.csv"
df = read_csv_smart(sample_csv)

required_cols = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0
r = df.iloc[ROW_IDX]
job_desc_text = f"""Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""
print("🧪 Prepared job_desc_text from row", ROW_IDX)

# ---------- 5) Upload the two CSVs to OpenAI so later cells can attach them ----------
client = OpenAI()  # API key already set in your Environment Setup cell
to_upload = [
    INPUTS_DIR / "Ground Truth Masterfile.csv",
    INPUTS_DIR / "Sample JDs.csv",
]
uploaded = []
for p in to_upload:
    with open(p, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
    uploaded.append(up)

file_ids = [u.id for u in uploaded]  # <-- used by the Responses API cell later
print("⬆️ Uploaded file_ids:", file_ids)

# ---------- 6) Write a small manifest so you can audit each run ----------
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "inputs": [str(p) for p in (INPUTS_DIR / "Ground Truth Masterfile.csv",
                                 INPUTS_DIR / "Sample JDs.csv")],
    "resources_dir": str(RESOURCES_DIR) if RESOURCES_DIR.exists() else None,
    "uploaded_file_ids": file_ids,
}
(RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n📁 Current run tree (first few entries):")
for i, p in enumerate(sorted(RUN_DIR.rglob("*"))):
    print(" -", p.relative_to(RUN_DIR))
    if i > 25:
        print(" … (truncated)")
        break

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗂️ Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Ground Truth Masterfile.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/inputs/Ground Truth Masterfile.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/Sample JDs.csv  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/inputs/Sample JDs.csv
📄 Copied: /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip  →  /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/inputs/MNPS_Prompt_Resources.zip
🧰 Unpacked resources to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/inputs/resources
✅ Read Sample JDs.csv with encoding=cp1252
🧪 Prepared job_desc_text from row 0


/tmp/ipython-input-1838845499.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


⬆️ Uploaded file_ids: ['file-4KiiGxx5HdNLY3PJGkimwe', 'file-CTovMCTTYteug1gG1su1x4']

📁 Current run tree (first few entries):
 - RUN_METADATA.json
 - inputs
 - inputs/Ground Truth Masterfile.csv
 - inputs/MNPS_Prompt_Resources.zip
 - inputs/Sample JDs.csv
 - inputs/resources
 - inputs/resources/Competency Extended Descriptions.csv
 - inputs/resources/Korn_Ferry Lominger 38 Competencies.csv
 - inputs/resources/MNPS KSACs.csv
 - inputs/resources/MNPS Roles.csv
 - outputs


In [4]:
# Cell 6
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


👍 gpt-4o-2024-11-20 is available.


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.

# **Version 6.0**
> Pulls data input fro mounted Google drive folder and unzips for use in /content/  

In [5]:
# Cell 8
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/My Drive/Colab Notebooks/Data Inputs/MNPS_Prompt_Resources.zip
  inflating: /content/Korn_Ferry Lominger 38 Competencies.csv  
  inflating: /content/Competency Extended Descriptions.csv  
  inflating: /content/MNPS KSACs.csv  
  inflating: /content/MNPS Roles.csv  


## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [6]:
#Cell 12
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

In [7]:
#Cell 14
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [8]:
# Cell 14.9 — Load MNPS Roles & KSACs (closed set)

from pathlib import Path
import pandas as pd, io, os, re
from collections import OrderedDict

# (Optional) set explicit paths if you know them; otherwise auto-detect:
ROLES_CSV  = globals().get("ROLES_CSV",  None)  # e.g., "/content/MNPS Roles.csv"
KSACS_CSV  = globals().get("KSACS_CSV",  None)  # e.g., "/content/MNPS KSACs.csv"

# --- version-safe reader (no `errors=` kw) ---
def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252"]
    for enc in encs:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    with open(path, "rb") as f:
        raw = f.read()
    for enc in encs + ["latin1"]:
        try:
            txt = raw.decode(enc, errors="ignore")
            return pd.read_csv(io.StringIO(txt))
        except Exception:
            pass
    return pd.read_csv(path, engine="python")

def _auto_find(*names):
    roots = [Path(globals().get("OUTPUTS_DIR",".")), Path.cwd(), Path("/content"), Path("/content/drive/My Drive")]
    hits = []
    for r in roots:
        try:
            for nm in names:
                for p in r.rglob(nm):
                    if p.is_file():
                        hits.append(p)
        except Exception:
            pass
    if not hits:
        return None
    # newest first, then shallower
    hits = sorted(hits, key=lambda p: (-p.stat().st_mtime, str(p).count(os.sep)))
    return str(hits[0])

def _pick_col(cols, *patterns):
    for c in cols:
        cl = c.lower()
        for pat in patterns:
            if re.search(pat, cl):
                return c
    return None

# --- Locate files ---
if not ROLES_CSV or not Path(str(ROLES_CSV)).exists():
    ROLES_CSV = _auto_find("MNPS Roles.csv") or _auto_find("MNPS_Roles.csv")
if not KSACS_CSV or not Path(str(KSACS_CSV)).exists():
    KSACS_CSV = _auto_find("MNPS KSACs.csv") or _auto_find("MNPS_KSACs.csv")

if not ROLES_CSV or not Path(ROLES_CSV).exists():
    raise FileNotFoundError("Could not locate 'MNPS Roles.csv'. Set ROLES_CSV to its path.")
if not KSACS_CSV or not Path(KSACS_CSV).exists():
    print("[Warn] 'MNPS KSACs.csv' not found — proceeding without KSAC blobs.")
    KSACS_CSV = None

# --- Load Roles ---
df_roles = _read_csv_robust(ROLES_CSV)
role_col = _pick_col(df_roles.columns, r"^role$", r"major.*role", r"major.*group", r"mnps.*role")
if not role_col:
    # fall back to first column
    role_col = df_roles.columns[0]

# Build VALID_ROLES (preserve first-seen order, drop blanks)
_valid_roles = []
seen = set()
for val in df_roles[role_col].astype(str).fillna("").tolist():
    name = val.strip()
    if not name:
        continue
    if name not in seen:
        seen.add(name)
        _valid_roles.append(name)

if not _valid_roles:
    raise ValueError("MNPS Roles file loaded but produced an empty role list.")

# --- Load KSACs (role -> concatenated KSAC text) ---
role_ksac = {}
if KSACS_CSV:
    df_ks = _read_csv_robust(KSACS_CSV)
    # try to identify columns
    ks_role_col = _pick_col(df_ks.columns, r"^role$", r"major.*role", r"role.*name")
    ks_text_cols = [c for c in df_ks.columns
                    if re.search(r"(ksac|knowledge|skills|abilities|competenc|description)", c, flags=re.I)]
    if not ks_role_col:
        # if we can't find a role column, bail gracefully
        ks_role_col = df_ks.columns[0]
    if not ks_text_cols:
        # fall back to all non-role columns
        ks_text_cols = [c for c in df_ks.columns if c != ks_role_col]

    # raw map by KSAC file's role labels
    tmp_map = {}
    for _, r in df_ks.iterrows():
        rk = str(r.get(ks_role_col, "") or "").strip()
        if not rk:
            continue
        parts = []
        for c in ks_text_cols:
            v = str(r.get(c, "") or "").strip()
            if v:
                parts.append(v)
        if not parts:
            continue
        blob = " ".join(parts)
        tmp_map.setdefault(rk, []).append(blob)

    # collapse lists
    tmp_map = {k: " ".join(vs) for k, vs in tmp_map.items()}

    # Align KSAC labels to the closed set names.
    # If KSAC uses variants like "Budget Partner" but "Partner" is the official role,
    # attach the KSAC text to the official role when it is a substring match.
    role_ksac = {r: "" for r in _valid_roles}
    for ks_label, blob in tmp_map.items():
        matched = False
        for official in _valid_roles:
            if official.lower() == ks_label.lower() or official.lower() in ks_label.lower() or ks_label.lower() in official.lower():
                role_ksac[official] = (role_ksac.get(official, "") + " " + blob).strip()
                matched = True
                break
        if not matched:
            # keep unmapped KSACs separate (rare); not harmful
            role_ksac[ks_label] = role_ksac.get(ks_label, "") + " " + blob

# --- Expose globals used by 15.2 and 16 ---
VALID_ROLES = _valid_roles  # closed set
globals()["VALID_ROLES"] = VALID_ROLES
globals()["role_ksac"] = role_ksac

print(f"[Closed Set] Loaded {len(VALID_ROLES)} roles from: {ROLES_CSV}")
print("  Example roles:", ", ".join(VALID_ROLES[:10]), ("..." if len(VALID_ROLES) > 10 else ""))
print(f"[Closed Set] KSAC blobs attached for {sum(bool(v) for v in role_ksac.values())} roles.")


[Closed Set] Loaded 64 roles from: /content/drive/MyDrive/Colab Notebooks/Run Results/RUN_20250924_200625/inputs/resources/MNPS Roles.csv
  Example roles: Accountant, Administrative Assistant, Advisor, Agent, Aide, Analyst, Architect (Technology-Focused), Associate, Assistant, Auditor ...
[Closed Set] KSAC blobs attached for 60 roles.


In [9]:
# ===== Cell 14.95 — Role lexicon & synonyms (augmented for the misses) =====

BASE_VALID_ROLES = list(globals().get("VALID_ROLES", []))
MUST_HAVE = ["Principal", "Assistant Principal", "Coach"]  # instructional (non-athletic)
VALID_ROLES = sorted(set(BASE_VALID_ROLES) | set(MUST_HAVE))
globals()["VALID_ROLES"] = VALID_ROLES

ROLE_SYNONYMS = dict(globals().get("ROLE_SYNONYMS", {}))
ROLE_SYNONYMS.update({
    # Teacher-like → Teacher
    "educator": "Teacher", "instructor": "Teacher",

    # Instructional Coach (non-athletic) aliases
    "instructional coach": "Coach", "academic coach": "Coach",
    "learning coach": "Coach", "teaching coach": "Coach",
    "teacher coach": "Coach", "coaching specialist": "Coach",
    # Restorative / Transition cues strongly map to instructional coaching
    "restorative practice": "Coach", "restorative practices": "Coach",
    "transition": "Coach",

    # Assistant Principal / Principal variants
    "asst principal": "Assistant Principal", "assistant-principal": "Assistant Principal",
    "associate principal": "Assistant Principal", "school principal": "Principal",

    # Common ED/Director variants (minor will carry the III later)
    "exec director": "Executive Director", "executive dir": "Executive Director",
    "senior director": "Director", "sr director": "Director",
})
globals()["ROLE_SYNONYMS"] = ROLE_SYNONYMS

# Disambiguation token sets (used by the injector)
LEADERSHIP_TOKENS = set("""
building leader admin leadership administrator leadership team
evaluations walkthroughs discipline parent relations
school improvement sip mtss budgeting staffing scheduling
compliance accreditation title ix 504 iep attendance safety drills
""".split())
AP_TOKENS = set("assistant principal ap dean discipline attendance testing coordinator".split())
PRINCIPAL_TOKENS = set("principal building principal head of school".split())
COACH_TOKENS = set("""
instructional coaching plc professional development pd model lessons co-teach
coaching cycles observation feedback data meetings curriculum alignment restorative transition
""".split())
ATHLETIC_TOKENS = set("athletic athletics sport sports varsity jv tournament game".split())

# Exec Director vs Director scope tokens
EXEC_DIR_TOKENS = set("""
district-wide portfolio cabinet superintendent board governance policy
systemwide cross-functional enterprise-wide multi-department
""".split())
DIRECTOR_TOKENS = set("department program unit team operations budget staff supervision compliance reporting".split())

globals().update({
    "LEADERSHIP_TOKENS": LEADERSHIP_TOKENS, "AP_TOKENS": AP_TOKENS, "PRINCIPAL_TOKENS": PRINCIPAL_TOKENS,
    "COACH_TOKENS": COACH_TOKENS, "ATHLETIC_TOKENS": ATHLETIC_TOKENS,
    "EXEC_DIR_TOKENS": EXEC_DIR_TOKENS, "DIRECTOR_TOKENS": DIRECTOR_TOKENS,
})
print("[14.95] VALID_ROLES:", ", ".join(sorted(VALID_ROLES)))
print("[14.95] Synonyms updated (Educator→Teacher; Restorative/Transition→Coach; AP/Principal; Exec Dir).")


[14.95] VALID_ROLES: Accountant, Administrative Assistant, Advisor, Agent, Aide, Analyst, Architect (Technology-Focused), Assistant, Assistant Principal, Associate, Auditor, Board Member, Buyer, Campus Support, Cashier, Chef, Chief, Clerk, Coach, Coordinator, Dean, Designer, Developer, Director, Dispatcher, Driver, Engineer, Executive  Officer, Executive Director, Facilitator, Foreman, Instructor, Intern, Interpreter, Laborer, Lead, Liaison, Librarian, Manager, Mechanic, Monitor, Officer, Operator, Para Pro, Partner, Principal, Receptionist, Registrar, Representative, School Counselor, School Principal, School Psychologist, Secretary, Skilled Laborer, Social Worker, Specialist, Substitute teacher, Supvervisor, Teacher, Technician, Therapist, Trainer, Translator, Tutor, Worker, Writer
[14.95] Synonyms updated (Educator→Teacher; Restorative/Transition→Coach; AP/Principal; Exec Dir).


In [10]:
# ===== Cell 14.951 — Translator lexicon augment =====
VALID_ROLES = sorted(set(globals().get("VALID_ROLES", [])) | {"Translator"})
globals()["VALID_ROLES"] = VALID_ROLES

ROLE_SYNONYMS = dict(globals().get("ROLE_SYNONYMS", {}))
ROLE_SYNONYMS.update({
    # Map common language-access variants to the role
    "translator": "Translator",
    "interpreter": "Translator",
    "bilingual translator": "Translator",
    "language access": "Translator",
    "parent outreach translator": "Translator",
    "family translator": "Translator",
})
globals()["ROLE_SYNONYMS"] = ROLE_SYNONYMS

# Token cues to detect translation as the *primary* function
TRANSLATOR_TOKENS = set("""
translate translation interpreter interpreting bilingual multilingual language access access to language
spanish arabic kurdish somali nepali amharic chinese french creole esl ell lep parent outreach families
""".split())

# Cues for liaison-type work so we can separate when translation is truly secondary
LIAISON_TOKENS = set("""
liaison outreach case management referral community partner engagement wraparound resource navigation
""".split())

globals().update({
    "TRANSLATOR_TOKENS": TRANSLATOR_TOKENS,
    "LIAISON_TOKENS": LIAISON_TOKENS,
})
print("[14.951] Added 'Translator' to closed set + synonyms and token cues.")


[14.951] Added 'Translator' to closed set + synonyms and token cues.


In [11]:
# ===== Cell 14.96 — Executive Director vs Director tokens + synonyms =====

# Ensure the closed set includes Assistant Principal explicitly
VALID_ROLES = sorted(set(globals().get("VALID_ROLES", [])) | {"Assistant Principal"})
globals()["VALID_ROLES"] = VALID_ROLES

ROLE_SYNONYMS = dict(globals().get("ROLE_SYNONYMS", {}))
ROLE_SYNONYMS.update({
    # Executive Director synonyms/variants
    "exec director": "Executive Director",
    "executive dir": "Executive Director",
    "ed ": "Executive Director",      # leading 'ed ' token (be careful but helpful)
    " ed": "Executive Director",      # trailing ' ed'
    "executive leadership": "Executive Director",
    # Director variants commonly seen
    "dir iii": "Director",   # minor level will carry the III
    "director iii": "Director",
    "sr director": "Director",
    "senior director": "Director",
    # Assistant Principal variants
    "ap ": "Assistant Principal", " ap": "Assistant Principal",
    "asst principal": "Assistant Principal",
    "assistant-principal": "Assistant Principal",
    "associate principal": "Assistant Principal",
    # Instructional Coach (non-athletic)
    "instructional coach": "Coach",
    "academic coach": "Coach",
    "learning coach": "Coach",
    "teaching coach": "Coach",
    "teacher coach": "Coach",
    "coaching specialist": "Coach",
})
globals()["ROLE_SYNONYMS"] = ROLE_SYNONYMS

# Lexicon to separate Exec Director vs Director (scope/authority signals)
EXEC_DIR_TOKENS = set("""
district-wide portfolio strategy cross-functional enterprise-wide
cabinet superintendent board policy strategic plan kpi governance ombuds
multi-department multi-program division head systemwide external relations
""".split())

DIRECTOR_TOKENS = set("""
department program unit team manage manage(s) oversees operational operations
budget staff supervision deliverables scheduling compliance reporting
""".split())

globals().update({
    "EXEC_DIR_TOKENS": EXEC_DIR_TOKENS,
    "DIRECTOR_TOKENS": DIRECTOR_TOKENS,
})
print("[14.96] Added Exec Director/Director synonyms & tokens. VALID_ROLES size:", len(VALID_ROLES))


[14.96] Added Exec Director/Director synonyms & tokens. VALID_ROLES size: 66


In [12]:
#Cell 15
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

In [13]:
# Cell 15.0 — Role Confidence Output Schema (v6.0)

from typing import List, Dict, Optional
try:
    from pydantic import BaseModel, Field
except Exception as e:
    raise ImportError("Pydantic is required. Install with: pip install pydantic") from e

class RoleConfidence(BaseModel):
    role: str = Field(..., description="One MNPS major role from the closed set.")
    confidence: float = Field(..., ge=0.0, le=1.0, description="Confidence in [0,1].")
    rationale: str = Field(..., description="Brief reason; cite determinant factors/KSACs.")

class RoleConfidenceTable(BaseModel):
    row_id: int = Field(..., description="Row id of the input record (0-based).")
    job_description_name: str = Field(..., description="Original Job Description Name for this row.")
    confidences: List[RoleConfidence] = Field(
        ..., description="Confidence for each MNPS role (ideally all roles)."
    )
    top_roles_summary: Optional[str] = Field(
        None, description="Optional 1-3 sentence summary of the top distinctions."
    )

# Pydantic v1 shim
if not hasattr(RoleConfidenceTable, "model_json_schema"):
    RoleConfidenceTable.model_json_schema = classmethod(lambda cls, *a, **k: cls.schema())


In [14]:
# ===== Cell 15.001 — Force shortlist conformance (optional) =====
STRICT_SHORTLIST = True   # require chosen role to come from the shortlist unless minima are not met
print("STRICT_SHORTLIST =", STRICT_SHORTLIST)


STRICT_SHORTLIST = True


In [15]:
# Cell 15.1 — Role Confidence Prompt Builder (v6.0)

import json
import textwrap

# Expect these exist from earlier cells; we will degrade gracefully.
VALID_ROLES = globals().get("VALID_ROLES", [])
role_ksac = globals().get("role_ksac", {})  # dict: role -> KSAC blob string

ROLE_CONFIDENCE_MODEL = globals().get("ROLE_CONFIDENCE_MODEL", globals().get("MODEL", "gpt-4o-2024-11-20"))
ROLE_CONFIDENCE_TEMP  = float(globals().get("ROLE_CONFIDENCE_TEMP", 0.2))

def _ksac_blurb_for_role(role: str) -> str:
    blob = role_ksac.get(role, "")
    return f"- {role}: {blob[:500]}{'...' if len(blob) > 500 else ''}"

def build_role_confidence_prompt(row, valid_roles=None):
    """Builds the prompt for the first API call (confidence by role)."""
    vr = valid_roles or VALID_ROLES
    if not vr:
        raise ValueError("VALID_ROLES is empty/undefined. Load MNPS Roles earlier.")

    # Determinant fields (robust access)
    def g(col): return str(row.get(col, "") or "")
    jd_name = g("Job Description Name")
    pos_sum = g("Position Summary")
    ess_fn  = g("Essential Functions")
    ksa     = g("Knowledge, Skills and Abilities")
    edu     = g("Education")
    exp     = g("Work Experience")
    lic     = g("Licenses and Certifications")

    ksac_section = "\n".join(_ksac_blurb_for_role(r) for r in vr[:64])

    instructions = f"""
You are evaluating one MNPS job against the CLOSED SET of MNPS major role groupings.

CLOSED SET (64 roles max, from MNPS Roles):
{", ".join(vr)}

KSAC Guidance (from MNPS KSACs; truncated where long):
{ksac_section}

Determinant Factors for this job:
- Job Description Name: {jd_name}
- Position Summary: {pos_sum}
- Essential Functions: {ess_fn}
- Knowledge, Skills and Abilities: {ksa}
- Education: {edu}
- Work Experience: {exp}
- Licenses and Certifications: {lic}

TASK:
For EACH role in the CLOSED SET, estimate a confidence in [0,1] for how well the job aligns to that role, based on KSAC fit and the determinant factors.
- Enforce eligibility minima implicitly: a role that clearly fails required Education/Experience/License should have very low confidence.
- Distinguish Analyst vs Specialist (analytics/metrics vs execution/process), Coordinator vs Technician, and Supervisor/Manager/Director by scope/people management/strategy.
- Keep rationales concise.

OUTPUT:
Return ONLY valid JSON that matches this JSON Schema:

{json.dumps(RoleConfidenceTable.model_json_schema(), indent=2)}
""".strip()

    return instructions


In [16]:
# ===== Cell 15.2 — Role Confidence pass (robust) =====
import os, io, json, math, re
from pathlib import Path
import pandas as pd

# ---- knobs ----
CONF_TOPK = int(globals().get("CONF_TOPK", 5))
CONF_MIN  = float(globals().get("CONF_MIN", 0.15))   # filter out ultra-low signals
MODEL     = globals().get("MODEL_CONFIDENCE", globals().get("MODEL", "gpt-4o-2024-11-20"))

# ---- IO resolve ----
OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

conf_top5_csv  = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_long_csv  = OUTPUTS_DIR / "role_confidence_indicators.csv"
conf_top5_json = OUTPUTS_DIR / "role_confidence_top5.json"

# ---- readers ----
def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]  # None = sniff
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    # last resort: decode+sniff
    raw = Path(path).read_bytes()
    for enc in encs:
        try:
            txt = raw.decode(enc, errors="ignore")
            for sep in seps:
                try:
                    df = pd.read_csv(io.StringIO(txt), sep=sep, engine="python")
                    if df.shape[1] >= 1:
                        return df
                except Exception:
                    pass
        except Exception:
            pass
    raise ValueError(f"Could not parse CSV: {path}")

def _auto_find_input():
    # find the latest likely “Sample JDs” style file if BATCH_INPUT_CSV is missing
    roots = [
        Path(globals().get("INPUTS_DIR",".")),
        Path.cwd(),
        Path("/content"),
        Path("/content/drive/My Drive/Colab Notebooks"),
    ]
    names = ["Sample JDs", "Sample_JDs", "New Sample", "SampleJDs"]
    cands = []
    for r in roots:
        if not r.exists(): continue
        try:
            for p in r.rglob("*.csv"):
                if any(n.lower() in p.name.lower() for n in names):
                    cands.append(p)
        except Exception:
            pass
    if not cands: return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0])

# ---- load input ----
if "BATCH_INPUT_CSV" in globals() and BATCH_INPUT_CSV and Path(BATCH_INPUT_CSV).exists():
    in_path = BATCH_INPUT_CSV
else:
    guess = _auto_find_input()
    if not guess:
        raise FileNotFoundError("BATCH_INPUT_CSV not set and no input CSV auto-found. Set BATCH_INPUT_CSV.")
    in_path = guess

in_df = _read_csv_robust(in_path)

# ensure row_id
if "row_id" not in in_df.columns:
    in_df = in_df.reset_index().rename(columns={"index":"row_id"})

# required cols check (we only warn; still proceed with what we have)
required = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing = [c for c in required if c not in in_df.columns]
if missing:
    print(f"[15.2] WARNING: Missing columns in input: {missing}. Continuing with whatever fields exist.")

# roles closed set
VALID_ROLES = globals().get("VALID_ROLES", [])
if not VALID_ROLES:
    raise ValueError("[15.2] VALID_ROLES is empty. Run 14.9/14.95 first to load MNPS Roles & KSACs.")

# canonicalizer (uses your synonyms if present)
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
def _canon_role(s: str) -> str:
    if not isinstance(s, str): s = str(s or "")
    raw = s.strip(); low = raw.lower()
    for k, v in ROLE_SYNONYMS.items():
        if k in low: return v
    for r in VALID_ROLES:
        if low == r.lower(): return r
    for r in sorted(VALID_ROLES, key=lambda x: -len(x)):
        if r.lower() in low or low in r.lower(): return r
    return raw.strip().title()

# ---- prompt builder for confidence ----
def _conf_prompt(row) -> str:
    def g(k):
        try:
            return str(row.get(k, "") if hasattr(row, "get") else row[k])
        except Exception:
            return ""
    role_list = ", ".join(sorted(set(VALID_ROLES)))
    return f"""
You are scoring *major role* fit for one MNPS job against the CLOSED set of roles.

Return STRICT JSON:
{{
  "candidates": [
    {{"role": "<one of [{role_list}]>", "confidence": 0.0_to_1.0, "rationale": "≤20 words"}},
    ...
  ]
}}

Rules:
- Consider **minimum bounds**: do NOT score a role high if the job clearly fails that role's minimum Education, Licenses/Certifications, or required years of Experience.
- Treat "Coach" as **instructional/non-athletic** only (PD, PLCs, coaching cycles). If duties are athletic-only, give "Coach" very low score.
- Assistant Principal vs Principal: prefer AP when duties are delegated ops/discipline/testing and not final evaluator/budget owner.
- Executive Director vs Director: Exec Dir only for cabinet/district-wide scope; otherwise Director (level handled in a later step).

Job:
- Job Description Name: {g('Job Description Name')}
- Position Summary: {g('Position Summary')}
- Essential Functions: {g('Essential Functions')}
- KSACs: {g('Knowledge, Skills and Abilities')}
- Education: {g('Education')}
- Work Experience: {g('Work Experience')}
- Licenses/Certifications: {g('Licenses and Certifications')}
""".strip()

# ---- model caller (uses your existing call_llm_json if present) ----
def _call_confidence(prompt: str) -> dict:
    # prefer your helper if defined
    if "call_llm_json" in globals():
        raw = call_llm_json(prompt, model=MODEL)
        try:
            return json.loads(raw)
        except Exception as e:
            raise ValueError(f"JSON parse error: {e}\nRaw: {raw[:400]}")
    # fallback: Responses API (parse-free text JSON)
    if "client" in globals():
        try:
            resp = client.responses.create(model=MODEL, input=[{"role":"user","content": prompt}])
            # scrape JSON from text
            txt = resp.output_text if hasattr(resp, "output_text") else str(resp)
            m = re.search(r"\{.*\}", txt, flags=re.S)
            if not m:
                raise ValueError(f"No JSON object found in response.\nFirst 400 chars:\n{txt[:400]}")
            return json.loads(m.group(0))
        except Exception as e:
            raise ValueError(f"Responses API error: {e}")
    raise RuntimeError("No model caller available (call_llm_json or client).")

# ---- run pass ----
rows_top5, rows_long, json_map = [], [], {}
n_ok, n_err = 0, 0

for _, row in in_df.iterrows():
    rid = row.get("row_id", _)
    try:
        prompt = _conf_prompt(row)
        obj = _call_confidence(prompt)
        cands = obj.get("candidates", []) if isinstance(obj, dict) else []
        # normalize & filter
        norm = []
        for c in cands:
            role = _canon_role(c.get("role",""))
            conf = c.get("confidence", None)
            try: conf = float(conf)
            except Exception: conf = None
            if role and (conf is None or conf >= CONF_MIN):
                norm.append({"role": role, "confidence": conf, "rationale": (c.get("rationale","") or "")[:120]})
        # sort desc by confidence (None to bottom)
        norm.sort(key=lambda d: (-999 if d["confidence"] is None else -float(d["confidence"])))
        top = norm[:CONF_TOPK] if CONF_TOPK > 0 else norm

        # --- write row entries ---
        # top5 row (wide-ish)
        top_row = {
            "row_id": rid,
            "Job Description Name": row.get("Job Description Name",""),
        }
        for i, d in enumerate(top, start=1):
            top_row[f"top{i}_role"] = d["role"]
            top_row[f"top{i}_confidence"] = d["confidence"]
        rows_top5.append(top_row)

        # long table rows (at least the ones we got back)
        for d in top:
            rows_long.append({
                "row_id": rid,
                "Job Description Name": row.get("Job Description Name",""),
                "role": d["role"],
                "confidence": d["confidence"],
                "rationale": d["rationale"],
            })

        # json map for injector/report
        json_map[int(rid)] = [{"role": d["role"], "confidence": d["confidence"], "rationale": d["rationale"]} for d in top]
        n_ok += 1
    except Exception as e:
        n_err += 1
        # still record a stub so files aren't empty
        rows_top5.append({
            "row_id": rid,
            "Job Description Name": row.get("Job Description Name",""),
        })
        json_map[int(rid)] = []
        # (no long rows for this record)

# ---- save artifacts ----
pd.DataFrame(rows_top5).to_csv(conf_top5_csv, index=False, encoding="utf-8")
# Always write *something* to the long file so it isn't 0/1-byte:
pd.DataFrame(rows_long if rows_long else [{"row_id": None, "Job Description Name": None, "role": None, "confidence": None, "rationale": None}]) \
  .to_csv(conf_long_csv, index=False, encoding="utf-8")
conf_top5_json.write_text(json.dumps(json_map, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"[15.2] Confidence pass complete — ok: {n_ok}  errors: {n_err}")
print(f"[15.2] Wrote: {conf_top5_csv.name} ({conf_top5_csv.stat().st_size} bytes)")
print(f"[15.2] Wrote: {conf_long_csv.name} ({conf_long_csv.stat().st_size} bytes)")
print(f"[15.2] Wrote: {conf_top5_json.name} ({conf_top5_json.stat().st_size} bytes)")


[15.2] Confidence pass complete — ok: 35  errors: 0
[15.2] Wrote: role_confidence_top5.csv (3764 bytes)
[15.2] Wrote: role_confidence_indicators.csv (26126 bytes)
[15.2] Wrote: role_confidence_top5.json (33846 bytes)


In [17]:
# ===== Cell 15.21 — Confidence files quick check (v6.0.1) =====
import json, glob, io
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES = {
    "long": ["role_confidence_indicators.csv"],
    "top5": ["role_confidence_top5.csv"],
    "json": ["role_confidence_top5.json"],
}

def _find_file(name_list):
    # look in outputs/, parent of outputs/, and CWD; then glob
    roots = [OUTPUTS_DIR, OUTPUTS_DIR.parent, Path.cwd()]
    for root in roots:
        for nm in name_list:
            p = root / nm
            if p.exists() and p.stat().st_size >= 1:
                return p
    for root in roots:
        for nm in name_list:
            hits = glob.glob(str(root / "**" / nm), recursive=True)
            if hits:
                hits = sorted(hits, key=lambda s: Path(s).stat().st_mtime, reverse=True)
                return Path(hits[0])
    return OUTPUTS_DIR / name_list[0]

# Define globals used by other cells
ROLE_CONF_CSV_LONG = _find_file(CANDIDATES["long"])
ROLE_CONF_CSV_TOP5 = _find_file(CANDIDATES["top5"])
ROLE_CONF_JSON     = _find_file(CANDIDATES["json"])
globals().update({
    "ROLE_CONF_CSV_LONG": ROLE_CONF_CSV_LONG,
    "ROLE_CONF_CSV_TOP5": ROLE_CONF_CSV_TOP5,
    "ROLE_CONF_JSON": ROLE_CONF_JSON,
})

for p in [ROLE_CONF_CSV_LONG, ROLE_CONF_CSV_TOP5, ROLE_CONF_JSON]:
    print(p.name, "→", p, "| exists:", p.exists(), "size:", (p.stat().st_size if p.exists() else 0))

def _read_csv_robust(path: Path):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        return pd.read_csv(path, engine="python")
    except Exception:
        raw = path.read_bytes()
        for enc in ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]:
            try:
                return pd.read_csv(io.StringIO(raw.decode(enc, errors="ignore")), engine="python")
            except Exception:
                pass
    return None

df_top5 = _read_csv_robust(ROLE_CONF_CSV_TOP5)
df_long = _read_csv_robust(ROLE_CONF_CSV_LONG)

print("\n[top5] role_confidence_top5.csv")
if df_top5 is not None:
    print(" rows:", len(df_top5), "| cols:", list(df_top5.columns)[:12])
    display(df_top5.head(5))
else:
    print(" not readable or empty")

print("\n[long] role_confidence_indicators.csv")
if df_long is not None:
    print(" rows:", len(df_long), "| cols:", list(df_long.columns)[:12])
    display(df_long.head(10))
else:
    print(" not readable or empty")

print("\n[json] role_confidence_top5.json")
if ROLE_CONF_JSON.exists():
    txt = ROLE_CONF_JSON.read_text(encoding="utf-8", errors="ignore")
    try:
        obj = json.loads(txt or "{}")
        non_empty = sum(1 for v in obj.values() if isinstance(v, list) and v)
        print(f" bytes={len(txt)} | keys={len(obj)} | non_empty_lists={non_empty}")
        # show one sample entry
        for k, v in obj.items():
            print(" sample:", k, "->", v[:3])
            break
    except Exception as e:
        print(" JSON parse error:", e)
else:
    print(" file not found")


role_confidence_indicators.csv → /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/role_confidence_indicators.csv | exists: True size: 26126
role_confidence_top5.csv → /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/role_confidence_top5.csv | exists: True size: 3764
role_confidence_top5.json → /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/role_confidence_top5.json | exists: True size: 33846

[top5] role_confidence_top5.csv
 rows: 35 | cols: ['row_id', 'Job Description Name', 'top1_role', 'top1_confidence', 'top2_role', 'top2_confidence', 'top3_role', 'top3_confidence', 'top4_role', 'top4_confidence', 'top5_role', 'top5_confidence']


,row_id,Job Description Name,top1_role,top1_confidence,top2_role,top2_confidence,top3_role,top3_confidence,top4_role,top4_confidence,top5_role,top5_confidence
0,0,Spec Transition,Coordinator,0.9,Specialist,0.85,Liaison,0.80,Facilitator,0.75,Advisor,0.7
1,1,Teacher CTE Criminal Justice,Teacher,1.0,Trainer,0.90,Specialist,0.80,Facilitator,0.60,Coach,0.5
2,2,Teacher Ex Ed Vision,Teacher,1.0,Specialist,0.70,Teacher,0.60,Trainer,0.40,Coordinator,0.3
3,3,Coach Advocacy Center,Coach,1.0,Specialist,0.85,Social Worker,0.75,Facilitator,0.70,Teacher,0.6
4,4,Teacher Ex Ed Life Skills,Teacher,1.0,Specialist,0.80,School Counselor,0.60,Trainer,0.60,Social Worker,0.4



[long] role_confidence_indicators.csv
 rows: 175 | cols: ['row_id', 'Job Description Name', 'role', 'confidence', 'rationale']


,row_id,Job Description Name,role,confidence,rationale
0,0,Spec Transition,Coordinator,0.90,"Facilitates supports, services, and compliance..."
1,0,Spec Transition,Specialist,0.85,"Specializes in transition services, IEP implem..."
2,0,Spec Transition,Liaison,0.80,"Acts as intermediary between schools, VR, and ..."
3,0,Spec Transition,Facilitator,0.75,Facilitates services related to career readine...
4,0,Spec Transition,Advisor,0.70,Supports and advises students on transition pa...
5,1,Teacher CTE Criminal Justice,Teacher,1.00,Job is directly teaching CTE Criminal Justice ...
6,1,Teacher CTE Criminal Justice,Trainer,0.90,Role involves educating others in a specialize...
7,1,Teacher CTE Criminal Justice,Specialist,0.80,Focus on a specialized field (Criminal Justice...
8,1,Teacher CTE Criminal Justice,Facilitator,0.60,"Some facilitation of student learning fits, bu..."
9,1,Teacher CTE Criminal Justice,Coach,0.50,Possible overlap with instructional coaching a...



[json] role_confidence_top5.json
 bytes=33846 | keys=35 | non_empty_lists=35
 sample: 0 -> [{'role': 'Coordinator', 'confidence': 0.9, 'rationale': 'Facilitates supports, services, and compliance tasks in transition programs; aligns with coordination of IEPs, VR servic'}, {'role': 'Specialist', 'confidence': 0.85, 'rationale': 'Specializes in transition services, IEP implementation, vocational tasks, aligning with deep, focused expertise rather t'}, {'role': 'Liaison', 'confidence': 0.8, 'rationale': 'Acts as intermediary between schools, VR, and employers to ensure alignment and service continuity for students.'}]


In [18]:
# ===== Cell 15.25 — Backfill confidence JSON from Top-5 CSV (canonical roles updated) =====
from pathlib import Path
import pandas as pd, json, math, re

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
conf_top5 = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"

VALID_ROLES = globals().get("VALID_ROLES", [])
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})

def canonical_role(s: str) -> str:
    if not isinstance(s, str): s = str(s or "")
    raw = s.strip(); low = raw.lower()
    # synonyms (substring friendly)
    for k, v in ROLE_SYNONYMS.items():
        if k in low: return v
    # exact
    for r in VALID_ROLES:
        if low == r.lower(): return r
    # substring to the longest matching label
    for r in sorted(VALID_ROLES, key=lambda x: -len(x)):
        if r.lower() in low or low in r.lower(): return r
    # keep "Athletic Coach" literal if it appears; we use it to *exclude* instructional coach
    if "athletic coach" in low: return "Athletic Coach"
    return raw.strip().title()

def _isnum(x):
    try: return not math.isnan(float(x))
    except Exception: return False

if not conf_top5.exists():
    raise FileNotFoundError(f"Top-5 CSV not found: {conf_top5}")

df = pd.read_csv(conf_top5)
conf_map = {}
for idx, r in df.iterrows():
    rid = r.get("row_id", idx)
    try: rid = int(rid)
    except Exception: rid = int(idx)
    entries = []
    for i in range(1, 6):
        role = canonical_role(r.get(f"top{i}_role", ""))
        conf = r.get(f"top{i}_confidence", None)
        conf = float(conf) if _isnum(conf) else None
        if role:
            entries.append({"role": role, "confidence": conf, "rationale": ""})
    conf_map[rid] = entries

conf_json.write_text(json.dumps(conf_map, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[Backfill] Wrote JSON map -> {conf_json} (records={len(conf_map)})")


[Backfill] Wrote JSON map -> /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/role_confidence_top5.json (records=35)


In [19]:
# Cell 15.31 — Build CONF_HINTS (id + name keyed) from Top-5 CSV/JSON
from pathlib import Path
import pandas as pd, json

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
conf_top5 = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"

CONF_HINTS = {}         # key: int(row_id) -> list[(role, conf)]
CONF_HINTS_BY_NAME = {} # key: "Job Description Name" (casefold)

def _read_top5():
    if conf_top5.exists():
        try:
            return pd.read_csv(conf_top5)
        except Exception:
            pass
    return pd.DataFrame()

def _read_json():
    if conf_json.exists():
        try:
            return json.loads(conf_json.read_text(encoding="utf-8") or "{}")
        except Exception:
            pass
    return {}

top5_df = _read_top5()
js_map  = _read_json()

# Build from JSON if present
for k, entries in js_map.items():
    try:
        rid = int(k)
        lst = []
        for d in entries or []:
            role = str(d.get("role","")).strip()
            conf = d.get("confidence", None)
            lst.append((role, conf))
        if lst:
            CONF_HINTS[rid] = lst
    except Exception:
        continue

# Also build from CSV (covers the case where JSON is empty)
if not top5_df.empty:
    name_col = None
    for c in top5_df.columns:
        if str(c).strip().lower() in {"job description name","job_title_original","original job title"}:
            name_col = c; break
    for _, r in top5_df.iterrows():
        rid = r.get("row_id")
        try: rid = int(rid)
        except Exception: rid = None

        pairs = []
        for i in range(1,6):
            role = str(r.get(f"top{i}_role","") or "").strip()
            conf = r.get(f"top{i}_confidence", None)
            try: conf = float(conf)
            except Exception: conf = None
            if role:
                pairs.append((role, conf))
        if pairs:
            if rid is not None:
                CONF_HINTS[rid] = pairs
            if name_col:
                nm = str(r.get(name_col,"")).strip().casefold()
                if nm:
                    CONF_HINTS_BY_NAME[nm] = pairs

print(f"[Hints] id-hints: {len(CONF_HINTS)} | name-hints: {len(CONF_HINTS_BY_NAME)}")


[Hints] id-hints: 35 | name-hints: 35


In [20]:
USE_CONF_SHORTLIST_IN_PROMPT = True   # turn injection on/off
CONF_K = 3                            # shortlist size
CONF_MIN = 0.20                       # ignore roles below this confidence
ENFORCE_MIN_BOUNDS_TEXT = True        # keep HARD FILTER text in prompt


In [21]:
# ===== Cell 15.35 — Confidence shortlist + policy injector (Leadership/Coach, Manager vs Supervisor, ExecDir vs Director) =====
import re
USE_CONF_SHORTLIST_IN_PROMPT = globals().get("USE_CONF_SHORTLIST_IN_PROMPT", True)
CONF_K   = int(globals().get("CONF_K", 3))
CONF_MIN = float(globals().get("CONF_MIN", 0.20))
ENFORCE_MIN_BOUNDS_TEXT = bool(globals().get("ENFORCE_MIN_BOUNDS_TEXT", True))
STRICT_SHORTLIST = bool(globals().get("STRICT_SHORTLIST", False))  # you may set True in a small cell above

ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
VALID_ROLES   = globals().get("VALID_ROLES", [])
LEADERSHIP_TOKENS = globals().get("LEADERSHIP_TOKENS", set())
AP_TOKENS = globals().get("AP_TOKENS", set()); PRINCIPAL_TOKENS = globals().get("PRINCIPAL_TOKENS", set())
COACH_TOKENS = globals().get("COACH_TOKENS", set()); ATHLETIC_TOKENS = globals().get("ATHLETIC_TOKENS", set())
EXEC_DIR_TOKENS = globals().get("EXEC_DIR_TOKENS", set()); DIRECTOR_TOKENS = globals().get("DIRECTOR_TOKENS", set())

def _get(row, k):
    try:
        return (row.get(k, "") if hasattr(row, "get") else row[k]) or ""
    except Exception:
        return ""

def _blob(row):
    return " ".join([
        _get(row,"Position Summary"), _get(row,"Essential Functions"),
        _get(row,"Knowledge, Skills and Abilities"), _get(row,"Education"),
        _get(row,"Work Experience"), _get(row,"Licenses and Certifications")
    ]).lower()

def _has_any(txt, toks): return any(t in txt for t in toks)

def _leadership_coach_block(row):
    t = _blob(row)
    hits = []
    if _has_any(t, LEADERSHIP_TOKENS) or _has_any(t, PRINCIPAL_TOKENS) or _has_any(t, AP_TOKENS):
        hits += [
            "### Leadership Disambiguation (Principal vs Assistant Principal)",
            "- **Eligibility minima (HARD FILTER):** administrator/leadership credential (as required) + stated years of teaching/leadership.",
            "- If both plausible, choose by scope:",
            "  • **Principal**: final evaluator, budget owner, whole-school authority.",
            "  • **Assistant Principal**: delegated ops/testing/discipline; reports to Principal.",
        ]
    if _has_any(t, COACH_TOKENS):
        hits += [
            "### Instructional Coach (non-athletic)",
            "- PD/PLCs, modeling lessons, coaching cycles, curriculum/data support → **Coach** (instructional).",
            "- **HARD FILTER:** requires active/prior teaching license + classroom experience.",
        ]
    if _has_any(t, ATHLETIC_TOKENS):
        hits += ["- Athletic terms detected — do **not** classify as instructional 'Coach' if duties are athletic-only."]
    return "\n".join(hits)

def _manager_supervisor_block(row):
    title = str(_get(row,"Job Description Name")).lower()
    t = _blob(row)
    cues_sup = any(w in title for w in ["mgr electronics","maintenance","repair","field","technician"]) or \
               re.search(r"\b(technicians?|crew|field|maintenance|repair)\b", t)
    if "mgr" in title or "manager" in title or "supervisor" in title or cues_sup:
        return "\n".join([
            "### Manager vs Supervisor Disambiguation",
            "- **Supervisor**: front-line oversight of hands-on staff (scheduling, assigning work, QA, training).",
            "- **Manager**: program/operations management (budgets, policy/process, multi-team outcomes).",
            "- If the duties emphasize direct oversight of technicians/maintenance/electronics crews → choose **Supervisor**.",
        ])
    return ""

def _execdir_director_block(row):
    t = _blob(row)
    if _has_any(t, EXEC_DIR_TOKENS) or _has_any(t, DIRECTOR_TOKENS):
        return "\n".join([
            "### Executive Director vs Director",
            "- **Executive Director** only when scope is district-wide/cabinet-level, policy/governance, or multi-department portfolios.",
            "- Otherwise choose **Director** and set level (I/II/III) by complexity/span; 'Director III' for large portfolios without cabinet scope.",
        ])
    return ""

def _shortlist_pairs(row):
    if not USE_CONF_SHORTLIST_IN_PROMPT: return []
    hints_by_id = globals().get("CONF_HINTS", {}); hints_by_nm = globals().get("CONF_HINTS_BY_NAME", {})
    rid = None
    try: rid = int(row.get("row_id", getattr(row,"name", 0)))
    except Exception: rid = None
    pairs = hints_by_id.get(rid, []) if rid is not None else []
    if not pairs:
        nm = _get(row,"Job Description Name").strip().casefold()
        if nm: pairs = hints_by_nm.get(nm, [])
    out=[]
    for role, conf in pairs:
        try: conf = float(conf) if conf is not None else None
        except Exception: conf = None
        if role and (conf is None or conf >= CONF_MIN): out.append((role, conf))
    return out[:CONF_K]

def _hint_block(row):
    blocks = []
    pairs = _shortlist_pairs(row)
    if pairs:
        blocks.append("### Confidence Shortlist (from cross-resource comparison)")
        for role, conf in pairs:
            blocks.append(f"- {role}" + ("" if conf is None else f" (confidence {conf:.2f})"))
        if STRICT_SHORTLIST:
            blocks.append("\nYou MUST choose from this shortlist unless all options fail eligibility minima. "
                          "If you deviate, begin your explanation with 'DEVIATION:' and say why.")
    if ENFORCE_MIN_BOUNDS_TEXT:
        blocks += ["",
            "**Minimum bounds policy (HARD FILTER):**",
            "- A role is INELIGIBLE if minimum Education, required Licenses/Certifications, or minimum Experience are not met.",
            "- If multiple eligible roles remain, prefer the highest-confidence shortlist role that best aligns to KSACs and duties.",
        ]
    for b in (_leadership_coach_block(row), _manager_supervisor_block(row), _execdir_director_block(row)):
        if b: blocks += ["", b]
    return "\n".join(blocks).strip()

# Base prompt builder fallback
def _base_prompt(row):
    zsp = (globals().get("zero_shot_prompt","") or "").strip() or \
          "Objective: Classify into a valid MNPS major role group and level (Lead/I/II/III) using KSACs and eligibility minima."
    g = lambda k: _get(row,k)
    return "\n".join([zsp,
        f"Job Description Name: {g('Job Description Name')}",
        f"Position Summary: {g('Position Summary')}",
        f"Essential Functions: {g('Essential Functions')}",
        f"Knowledge, Skills and Abilities: {g('Knowledge, Skills and Abilities')}",
        f"Education: {g('Education')}",
        f"Work Experience: {g('Work Experience')}",
        f"Licenses and Certifications: {g('Licenses and Certifications')}"]).strip()

# Store the original full_text_for_row in a different variable name
_original_full_text_builder = None
if 'full_text_for_row' in globals():
    _original_full_text_builder = full_text_for_row
elif 'build_prompt' in globals():
    def _original_full_text_builder(row):
        try: p,_ = build_prompt(row); return str(p)
        except Exception: return _base_prompt(row)
else:
    _original_full_text_builder = _base_prompt

def full_text_for_row(row):
    base = _original_full_text_builder(row)
    hint = _hint_block(row)
    return base + ("\n\n" + hint if hint else "")

print("[15.35] Injector ready: shortlist + Leadership/Coach + Manager vs Supervisor + ExecDir vs Director + Minima.")

[15.35] Injector ready: shortlist + Leadership/Coach + Manager vs Supervisor + ExecDir vs Director + Minima.


In [22]:
# ===== Cell 15.351 — Safe injector rebuild (no recursion, NaN-safe) =====
# Rebuilds full_text_for_row deterministically (no wrapping), includes:
# - Confidence shortlist (CONF_HINTS / CONF_HINTS_BY_NAME)
# - Minimum-bounds HARD FILTER text
# - Leadership: Principal vs Assistant Principal
# - Instructional Coach (non-athletic) vs Athletic
# - Manager vs Supervisor
# - Executive Director vs Director
# - Translator vs Specialist/Liaison (primary translation function)
# This avoids recursion errors from layered wrappers and is NaN-safe.

import re, math, json
import pandas as pd

# ---- knobs / globals (respected if already set) ----
USE_CONF_SHORTLIST_IN_PROMPT = globals().get("USE_CONF_SHORTLIST_IN_PROMPT", True)
CONF_K   = int(globals().get("CONF_K", 3))
CONF_MIN = float(globals().get("CONF_MIN", 0.20))
ENFORCE_MIN_BOUNDS_TEXT = bool(globals().get("ENFORCE_MIN_BOUNDS_TEXT", True))
STRICT_SHORTLIST = bool(globals().get("STRICT_SHORTLIST", False))

# Lexicons (default to empty sets if not defined elsewhere)
LEADERSHIP_TOKENS = globals().get("LEADERSHIP_TOKENS", set())
AP_TOKENS         = globals().get("AP_TOKENS", set())
PRINCIPAL_TOKENS  = globals().get("PRINCIPAL_TOKENS", set())
COACH_TOKENS      = globals().get("COACH_TOKENS", set())
ATHLETIC_TOKENS   = globals().get("ATHLETIC_TOKENS", set())
EXEC_DIR_TOKENS   = globals().get("EXEC_DIR_TOKENS", set())
DIRECTOR_TOKENS   = globals().get("DIRECTOR_TOKENS", set())
TRANSLATOR_TOKENS = globals().get("TRANSLATOR_TOKENS", set())
LIAISON_TOKENS    = globals().get("LIAISON_TOKENS", set())

# Confidence maps created earlier (15.2/15.25/15.31)
CONF_HINTS          = globals().get("CONF_HINTS", {})
CONF_HINTS_BY_NAME  = globals().get("CONF_HINTS_BY_NAME", {})

# ---- NaN-safe helpers ----
def _to_str(v):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v) if v is not None else ""

def _get(row, key):
    try:
        if hasattr(row, "get"):
            return _to_str(row.get(key, ""))
        # if it's a Series
        return _to_str(row[key]) if key in row else ""
    except Exception:
        return ""

def _blob(row):
    return " ".join([
        _get(row, "Job Description Name"),
        _get(row, "Position Summary"),
        _get(row, "Essential Functions"),
        _get(row, "Knowledge, Skills and Abilities"),
        _get(row, "Education"),
        _get(row, "Work Experience"),
        _get(row, "Licenses and Certifications"),
    ]).lower()

def _has_any(text, toks):
    return any(tok in text for tok in toks)

# ---- Policy hint blocks ----
def _leadership_block(row):
    t = _blob(row)
    if not (_has_any(t, LEADERSHIP_TOKENS) or _has_any(t, PRINCIPAL_TOKENS) or _has_any(t, AP_TOKENS)):
        return ""
    return "\n".join([
        "### Leadership Disambiguation (Principal vs Assistant Principal)",
        "- **HARD FILTER:** administrator/leadership credential (as required) + stated years of teaching/leadership.",
        "- If both plausible, choose by scope:",
        "  • **Principal**: final evaluator, budget owner, whole-school authority.",
        "  • **Assistant Principal**: delegated ops/testing/discipline; reports to Principal.",
    ])

def _coach_block(row):
    t = _blob(row)
    lines = []
    if _has_any(t, COACH_TOKENS):
        lines += [
            "### Instructional Coach (non-athletic)",
            "- PD/PLCs, modeling lessons, coaching cycles, curriculum/data support → choose **Coach** (instructional).",
            "- **HARD FILTER:** requires active/prior teaching license + classroom experience.",
        ]
    if _has_any(t, ATHLETIC_TOKENS):
        lines += ["- Athletic terms detected — do **not** classify as instructional 'Coach' if duties are athletic-only."]
    return "\n".join(lines)

def _mgr_vs_sup_block(row):
    title = _get(row, "Job Description Name").lower()
    t = _blob(row)
    cues_sup = any(w in title for w in ["mgr electronics","maintenance","repair","field","technician"]) or \
               re.search(r"\b(technicians?|crew|field|maintenance|repair)\b", t or "")
    if "mgr" in title or "manager" in title or "supervisor" in title or cues_sup:
        return "\n".join([
            "### Manager vs Supervisor",
            "- **Supervisor**: front-line oversight of hands-on staff (scheduling, assigning work, QA, training).",
            "- **Manager**: program/operations management (budgets, policy/process, multi-team outcomes).",
            "- If duties emphasize direct oversight of technicians/maintenance/electronics crews → choose **Supervisor**.",
        ])
    return ""

def _execdir_block(row):
    t = _blob(row)
    if not (_has_any(t, EXEC_DIR_TOKENS) or _has_any(t, DIRECTOR_TOKENS)):
        return ""
    return "\n".join([
        "### Executive Director vs Director",
        "- **Executive Director** only when scope is district-wide/cabinet-level, policy/governance, or multi-department portfolios.",
        "- Otherwise choose **Director** and set level (I/II/III) by complexity/span; 'Director III' for large portfolios without cabinet scope.",
    ])

def _translator_block(row):
    t = _blob(row)
    # translation primary: translator/interpreter terms or explicit language access
    hit_trans = _has_any(t, TRANSLATOR_TOKENS) or bool(re.search(r"\btranslator|interpreter\b", t or ""))
    if not hit_trans:
        return ""
    return "\n".join([
        "### Translator vs Specialist/Liaison",
        "- If translation/interpretation and language access for families/staff is the **primary** function → choose **Translator**.",
        "- If translation is clearly secondary to case management/community navigation, consider **Liaison/Advisor** instead.",
        "- **HARD FILTER:** required language proficiency/certification must be met for **Translator**.",
    ])

def _shortlist_pairs(row):
    if not USE_CONF_SHORTLIST_IN_PROMPT:
        return []
    # Prefer row_id lookup
    rid = None
    try:
        rid = int(row.get("row_id", getattr(row, "name", 0)))
    except Exception:
        rid = None
    pairs = CONF_HINTS.get(rid, []) if rid is not None else []
    if not pairs:
        nm = _get(row, "Job Description Name").strip().casefold()
        if nm:
            pairs = CONF_HINTS_BY_NAME.get(nm, [])
    out = []
    for role, conf in pairs:
        try:
            conf = float(conf) if conf is not None else None
        except Exception:
            conf = None
        if role and (conf is None or conf >= CONF_MIN):
            out.append((role, conf))
    return out[:CONF_K]

def _hint_block(row):
    blocks = []
    # shortlist
    pairs = _shortlist_pairs(row)
    if pairs:
        blocks.append("### Confidence Shortlist (from cross-resource comparison)")
        for role, conf in pairs:
            blocks.append(f"- {role}" + ("" if conf is None else f" (confidence {conf:.2f})"))
        if STRICT_SHORTLIST:
            blocks.append("\nYou MUST choose from this shortlist unless all options fail eligibility minima. "
                          "If you deviate, begin explanation with 'DEVIATION:' and say why.")
    # minima
    if ENFORCE_MIN_BOUNDS_TEXT:
        blocks += ["",
            "**Minimum bounds policy (HARD FILTER):**",
            "- A role is INELIGIBLE if minimum Education, required Licenses/Certifications, or minimum Experience are not met.",
            "- If multiple eligible roles remain, prefer the highest-confidence shortlist role that best aligns to KSACs and duties.",
        ]
    # policy disambiguators
    for b in (_leadership_block(row), _coach_block(row), _mgr_vs_sup_block(row), _execdir_block(row), _translator_block(row)):
        if b:
            blocks += ["", b]
    return "\n".join(blocks).strip()

# ---- Base prompt builder ----
def _base_prompt(row):
    zsp = (globals().get("zero_shot_prompt","") or "").strip() or \
          "Objective: Classify into a valid MNPS major role group and level (Lead/I/II/III) using KSACs and eligibility minima."
    return "\n".join([
        zsp,
        f"Job Description Name: {_get(row,'Job Description Name')}",
        f"Position Summary: {_get(row,'Position Summary')}",
        f"Essential Functions: {_get(row,'Essential Functions')}",
        f"Knowledge, Skills and Abilities: {_get(row,'Knowledge, Skills and Abilities')}",
        f"Education: {_get(row,'Education')}",
        f"Work Experience: {_get(row,'Work Experience')}",
        f"Licenses and Certifications: {_get(row,'Licenses and Certifications')}",
    ]).strip()

# ---- Final builder (no wrapping; deterministic) ----
def full_text_for_row(row):
    base = _base_prompt(row)
    hints = _hint_block(row)
    return base + ("\n\n" + hints if hints else "")

# Set a stable base reference so future cells can rely on it without wrapping
globals()["BASE_full_text_for_row"] = full_text_for_row

print("[15.351] Installed safe injector builder (no recursion, NaN-safe).")


[15.351] Installed safe injector builder (no recursion, NaN-safe).


In [23]:
# Cell 15.36 — Hints coverage & prompt tail sampler (self-contained)

import pandas as pd, io, os
from pathlib import Path

# ---------- robust readers & finders ----------
def _read_csv_robust(path: str) -> pd.DataFrame:
    encs = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]  # None = sniff
    for enc in encs:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                pass
    # manual decode + sniff
    raw = Path(path).read_bytes()
    for enc in encs:
        try:
            txt = raw.decode(enc, errors="ignore")
            for sep in seps:
                try:
                    df = pd.read_csv(io.StringIO(txt), sep=sep, engine="python")
                    if df.shape[1] >= 1:
                        return df
                except Exception:
                    pass
        except Exception:
            pass
    raise ValueError(f"Could not parse CSV: {path}")

def _auto_find_input():
    names = ["Sample JDs", "New Sample", "Sample_JDs", "Job_Classifications_Batch"]
    roots = [
        Path(globals().get("INPUTS_DIR",".")),
        Path.cwd(),
        Path("/content"),
        Path("/content/drive/My Drive/Colab Notebooks"),
    ]
    cands = []
    for r in roots:
        if not r.exists():
            continue
        try:
            for p in r.rglob("*.csv"):
                if any(n.lower() in p.name.lower() for n in names):
                    cands.append(p)
        except Exception:
            pass
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return str(cands[0])

# ---------- ensure in_df exists ----------
if "in_df" not in globals() or not isinstance(in_df, pd.DataFrame):
    if "BATCH_INPUT_CSV" in globals() and BATCH_INPUT_CSV and Path(BATCH_INPUT_CSV).exists():
        in_df = _read_csv_robust(BATCH_INPUT_CSV)
        print(f"[15.36] Loaded in_df from BATCH_INPUT_CSV: {BATCH_INPUT_CSV}")
    elif "df" in globals() and isinstance(df, pd.DataFrame):
        in_df = df.copy()
        print("[15.36] Using existing 'df' as in_df")
    else:
        guess = _auto_find_input()
        if guess:
            in_df = _read_csv_robust(guess)
            print(f"[15.36] Auto-found input CSV: {guess}")
        else:
            raise RuntimeError("Could not create in_df — set BATCH_INPUT_CSV or run your Inputs/Load cell first.")

# add row_id if missing
if "row_id" not in in_df.columns:
    in_df = in_df.reset_index().rename(columns={"index":"row_id"})

# ---------- verify injector is in place ----------
if "full_text_for_row" not in globals():
    raise RuntimeError("full_text_for_row not found. Run Cell 15.35 first (the injector).")

# ---------- coverage summary ----------
n_id = len(globals().get("CONF_HINTS", {}))
n_name = len(globals().get("CONF_HINTS_BY_NAME", {}))
print(f"[Hints] Coverage — by row_id: {n_id}, by job name: {n_name}")

# ---------- show prompt tails for a few rows ----------
SAMPLE_ROWS = [0, 1, 2]  # tweak as you like
for ridx in SAMPLE_ROWS:
    if ridx >= len(in_df):
        continue
    row = in_df.iloc[ridx].copy()
    if "row_id" not in row or pd.isna(row["row_id"]):
        row["row_id"] = ridx

    txt = full_text_for_row(row)
    tail = "\n".join(txt.splitlines()[-40:])
    print("="*80)
    print(f"ROW {int(row['row_id'])} — {row.get('Job Description Name','(no name)')}")
    print(tail)


[Hints] Coverage — by row_id: 35, by job name: 35
ROW 0 — Spec Transition
Understanding of Transition services and how office of vocational rehab operates, collaboration with school staff and outside agencies
Education: Bachelor's Degree In related field Preferred
Work Experience: 4-6 years Working with students in the area of Transition, work based learning, career exploration Required
Licenses and Certifications:

### Confidence Shortlist (from cross-resource comparison)
- Coordinator (confidence 0.90)
- Specialist (confidence 0.85)
- Liaison (confidence 0.80)

You MUST choose from this shortlist unless all options fail eligibility minima. If you deviate, begin explanation with 'DEVIATION:' and say why.

**Minimum bounds policy (HARD FILTER):**
- A role is INELIGIBLE if minimum Education, required Licenses/Certifications, or minimum Experience are not met.
- If multiple eligible roles remain, prefer the highest-confidence shortlist role that best aligns to KSACs and duties.

### Lead

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

# **Version 6.0**
> Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR  
> Composes text-only input (no attachments)  
> Newer SDK: server-enforced  
> Structured Outputs via parse; uses jsons  
> Save outputs to OUTPUTS_DIR


In [24]:
# ===== Cell 16 — Responses API (TEXT-ONLY, no attachments), saves to OUTPUTS_DIR =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, inspect

client = OpenAI()  # API key from Environment Setup

# ---- Requires earlier cells ----
assert 'MODEL_ID' in globals(), "Run Environment Setup first (MODEL_ID)."
assert 'INPUTS_DIR' in globals() and 'OUTPUTS_DIR' in globals(), "Run the unique-run Cell 4 first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt in your prompt cell."
assert 'job_desc_text' in globals(), "Cell 4 builds job_desc_text (row 0 smoke test)."

print("🤖 Using model:", MODEL_ID)

# If read_csv_smart exists (Cell 4), use it for robust encodings; else default to utf-8
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- Build a SMALL context from Ground Truth (first 3 rows) ----
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv"
context_block = ""
if gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        # keep only lightweight columns if present
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        # represent as compact JSON so the model can parse easily
        context_block = "Context (Ground Truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
    except Exception as e:
        context_block = f"Context note: Ground Truth CSV present but could not be summarized ({e})."

# ---- Compose text-only input (no attachments) ----
# Tip: the zero_shot_prompt you wrote mentions "attached reference sources" — we add a Context block instead.
full_text = (
    zero_shot_prompt.strip()
    + "\n\n"
    + (context_block + "\n\n" if context_block else "")
    + "Classify the following job description:\n\n"
    + job_desc_text
)

# ---- Capability detection for your SDK version ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

parsed = None
raw_text = ""

try:
    if supports_parse_schema:
        # Newer SDK: server-enforced Structured Outputs via parse()
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format=JobClassificationTable,  # Pydantic schema (Cells 14–15)
        )
        parsed  = resp.output_parsed
        raw_text = resp.output_text or ""
    elif supports_create_schema:
        # Mid SDK: enforce via create() + json_schema
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": full_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
            response_format={
                "type": "json_schema",
                "json_schema": {"name": "JobClassificationTable", "schema": schema, "strict": True},
            },
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
    else:
        # Old SDK: prompt-only enforcement + client-side validation
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict_text = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            "JSON Schema:\n" + schema_json + "\n\n"
            "Task:\n" + full_text
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": strict_text}]}],
            temperature=0.2,
            max_output_tokens=1400,
        )
        raw_text = getattr(resp, "output_text", None) or ""
        # Clean the raw_text to remove potential markdown formatting
        if raw_text.strip().startswith("```json"):
            raw_text = raw_text.strip()[7:].strip("`")
        parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None
except Exception as e:
    print("❗ Unexpected Responses API error:", e)
    raise

# ---- Save outputs to OUTPUTS_DIR ----
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
raw_path = Path(OUTPUTS_DIR) / "Raw_Response_SINGLE.json"   # was Raw_Response.json

if parsed is not None:
    rows = [row.model_dump() for row in parsed.job_classification_table]
    out_csv = Path(OUTPUTS_DIR) / "Job_Classifications_SINGLE.csv"   # was Job_Classifications.csv
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8")
    out_txt = Path(OUTPUTS_DIR) / "Narrative_SINGLE.txt"             # was Narrative.txt
    out_txt.write_text(parsed.narrative_rationale, encoding="utf-8")
    print("✅ Saved:", out_csv)
    print("✅ Saved:", out_txt)
else:
    print("⚠️ No parsed object returned; saved Raw_Response_SINGLE.json only at:", raw_path)

print("✅ Saved:", raw_path)

# ---- Console visibility ----
print("\n=== RAW JSON STRING FROM MODEL ===")
print(raw_text or "(empty)")
if parsed is not None:
    print("\n=== PARSED (Pydantic) ===")
    print(parsed.model_dump_json(indent=2))

# ---- List run outputs ----
print("\nContents of OUTPUTS_DIR:")
for p in sorted(Path(OUTPUTS_DIR).glob("*")):
    print(" -", p.name)

🤖 Using model: gpt-4o-2024-11-20
✅ Read Ground Truth Masterfile.csv with encoding=cp1252
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_SINGLE.csv
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Narrative_SINGLE.txt
✅ Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Raw_Response_SINGLE.json

=== RAW JSON STRING FROM MODEL ===

{
  "job_classification_table": [
    {
      "job_title_original": "Transition Specialist",
      "new_job_title": "Transition Support Specialist II",
      "major_role_group": "Specialist",
      "minor_sub_group": "II",
      "grouping_justification": "The role involves specialized support for students with disabilities in the area of transition services, including job readiness, acquisition, and retention. The responsibilities require a higher level of expertise and experience (4-6 years) compared to entry-level roles, b

# **Verion 6.0**
>  Pre-flight: are all prerequisites loaded for batch


In [25]:
# ===== Cell 16.44 — Prompt debug sampler (verify injection) =====
import pandas as pd

# pick a few rows to inspect
SAMPLE_ROWS = [0, 1, 2]   # change as needed

# Need the same input df the batch will use
if "in_df" not in globals() or not isinstance(in_df, pd.DataFrame):
    raise RuntimeError("in_df not found. Run your Inputs/Load cell first so in_df exists.")

for ridx in SAMPLE_ROWS:
    if ridx >= len(in_df):
        continue
    row = in_df.iloc[ridx].copy()
    # Make sure row has a row_id consistent with confidence maps
    if "row_id" not in row or pd.isna(row["row_id"]):
        row["row_id"] = ridx
    print("="*80)
    print(f"ROW {ridx} — {row.get('Job Description Name','(no name)')}")
    # Use the SAME function the batch uses
    if 'full_text_for_row' not in globals():
        raise RuntimeError("full_text_for_row is not defined. Ensure Cell 15.35 ran.")
    txt = full_text_for_row(row)
    # show only the tail to confirm injected block (adjust as you wish)
    tail = "\n".join(txt.splitlines()[-40:])
    print(tail)


ROW 0 — Spec Transition
Understanding of Transition services and how office of vocational rehab operates, collaboration with school staff and outside agencies
Education: Bachelor's Degree In related field Preferred
Work Experience: 4-6 years Working with students in the area of Transition, work based learning, career exploration Required
Licenses and Certifications:

### Confidence Shortlist (from cross-resource comparison)
- Coordinator (confidence 0.90)
- Specialist (confidence 0.85)
- Liaison (confidence 0.80)

You MUST choose from this shortlist unless all options fail eligibility minima. If you deviate, begin explanation with 'DEVIATION:' and say why.

**Minimum bounds policy (HARD FILTER):**
- A role is INELIGIBLE if minimum Education, required Licenses/Certifications, or minimum Experience are not met.
- If multiple eligible roles remain, prefer the highest-confidence shortlist role that best aligns to KSACs and duties.

### Leadership Disambiguation (Principal vs Assistant Prin

In [26]:
# ===== Cell 16.45 — Pre-flight: are all prerequisites loaded for batch? =====
from pathlib import Path

print("Have MODEL_ID:", 'MODEL_ID' in globals(), (MODEL_ID if 'MODEL_ID' in globals() else None))
print("Have df:", 'df' in globals(), (len(df) if 'df' in globals() else None))
print("Have zero_shot_prompt:", 'zero_shot_prompt' in globals())
print("Have OUTPUTS_DIR:", 'OUTPUTS_DIR' in globals(), (OUTPUTS_DIR if 'OUTPUTS_DIR' in globals() else None))

if 'RUN_DIR' in globals():
    print("RUN_DIR:", RUN_DIR)
    print("Outputs path will be:", Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv")
else:
    print("RUN_DIR missing — re-run your unique run cell (Cell 4).")


Have MODEL_ID: True gpt-4o-2024-11-20
Have df: True 35
Have zero_shot_prompt: True
Have OUTPUTS_DIR: True /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs
RUN_DIR: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625
Outputs path will be: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch.csv


In [27]:
from pathlib import Path
(Path(OUTPUTS_DIR)/"Job_Classifications_Batch.csv").unlink(missing_ok=True)
(Path(OUTPUTS_DIR)/"Batch_Errors.json").unlink(missing_ok=True)


# **Version 6.0**
>  Runs batch file  
> Batch v3.1 (DEBUG: loud logs, resume-safe, JSON fence fix)  
> Live peek into processing


In [28]:
# ===== Cell 16.5 — Batch v3.1 (DEBUG: loud logs, resume-safe, JSON fence fix) =====
from openai import OpenAI
from pathlib import Path
import pandas as pd, json, time, random, inspect, re, shutil

print("=== Batch v3.1 start ===")

# ---- prerequisites ----
assert 'df' in globals(), "Run Cell 4 first (loads df)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first."
assert 'MODEL_ID' in globals(), "Run Environment Setup first."
assert 'zero_shot_prompt' in globals(), "Define zero_shot_prompt (your prompt cell)."

print("MODEL_ID:", MODEL_ID)
print("Rows in df:", len(df))
print("OUTPUTS_DIR:", OUTPUTS_DIR)

# If available, show SDK version
try:
    import openai as _o
    print("openai SDK:", getattr(_o, "__version__", "(unknown)"))
except Exception:
    pass

client = OpenAI(timeout=60.0, max_retries=2)

# ---- robust CSV reader if you have it from Cell 4 ----
def _read_csv(path: Path, **kw):
    if 'read_csv_smart' in globals():
        return read_csv_smart(path, **kw)
    return pd.read_csv(path, encoding="utf-8", **kw)

# ---- tiny context from Ground Truth (once) ----
context_block = ""
gt_path = Path(INPUTS_DIR) / "Ground Truth Masterfile.csv" if 'INPUTS_DIR' in globals() else None
if gt_path and gt_path.exists():
    try:
        gt_df = _read_csv(gt_path).fillna("")
        preferred_cols = [
            "Original Job Title","New Job Title","Major Role Group","Minor Sub-Group","Justification for Grouping",
            "Position Summary","Education","Work Experience","Licenses and Certifications","Essential Functions","Knowledge, Skills and Abilities"
        ]
        cols = [c for c in preferred_cols if c in gt_df.columns] or list(gt_df.columns)[:8]
        mini = gt_df[cols].head(3)
        context_block = "Context (3 ground-truth examples):\n" + mini.to_json(orient="records", force_ascii=False)
        print("Context block chars:", len(context_block))
    except Exception as e:
        context_block = f"(Context unavailable: {e})"
        print("Context build error:", e)
else:
    print("No Ground Truth CSV found at", gt_path)

def build_job_text(r):
    def getv(col):
        try:
            v = r[col]
            return "" if pd.isna(v) else str(v)
        except Exception:
            return ""
    return f"""Job Description Name: {getv('Job Description Name')}

Position Summary: {getv('Position Summary')}
Education: {getv('Education')}
Work Experience: {getv('Work Experience')}
Licenses and Certifications: {getv('Licenses and Certifications')}
Essential Functions: {getv('Essential Functions')}
Knowledge, Skills and Abilities: {getv('Knowledge, Skills and Abilities')}
"""

def full_text_for_row(r):
    return (
        zero_shot_prompt.strip()
        + ("\n\n" + context_block if context_block else "")
        + "\n\nClassify the following job description:\n\n"
        + build_job_text(r)
    )

# ---- capability detection ----
def _has_param(obj, name: str) -> bool:
    try:
        return name in inspect.signature(obj).parameters
    except Exception:
        return False

supports_parse_schema  = _has_param(client.responses.parse,  "response_format")
supports_create_schema = _has_param(client.responses.create, "response_format")

print("supports_parse_schema:", supports_parse_schema,
      "| supports_create_schema:", supports_create_schema)

# ---- JSON sanitizers (strip ```json fences etc.) ----
_fence_re = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.DOTALL|re.IGNORECASE)
_brace_re = re.compile(r"\{.*\}", re.DOTALL)

def coerce_to_json_str(raw: str) -> str:
    if not isinstance(raw, str):
        return ""
    s = raw.strip()
    m = _fence_re.match(s)
    if m:
        s = m.group(1).strip()
    if not s.startswith("{"):
        m2 = _brace_re.search(s)
        if m2:
            s = m2.group(0)
    return s

# ---- call wrapper ----
def call_model_with_text(text, temp, max_tokens):
    if supports_parse_schema:
        resp = client.responses.parse(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format=JobClassificationTable,
        )
        return resp.output_parsed, resp.output_text or ""
    elif supports_create_schema:
        schema = JobClassificationTable.model_json_schema()
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role": "user", "content": [{"type":"input_text","text": text}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
            response_format={
                "type":"json_schema",
                "json_schema":{"name":"JobClassificationTable","schema":schema,"strict":True},
            },
        )
        raw = getattr(resp, "output_text", None) or ""
        try:
            return JobClassificationTable.model_validate_json(raw), raw
        except Exception:
            cleaned = coerce_to_json_str(raw)
            return JobClassificationTable.model_validate_json(cleaned), cleaned
    else:
        schema_json = json.dumps(JobClassificationTable.model_json_schema(), indent=2)
        strict = (
            "You MUST return ONLY valid JSON that matches the following JSON Schema. No prose, no markdown.\n"
            f"JSON Schema:\n{schema_json}\n\nTask:\n{text}"
        )
        resp = client.responses.create(
            model=MODEL_ID,
            input=[{"role":"user","content":[{"type":"input_text","text": strict}]}],
            temperature=temp,
            max_output_tokens=max_tokens,
        )
        raw = getattr(resp, "output_text", None) or ""
        cleaned = coerce_to_json_str(raw)
        return JobClassificationTable.model_validate_json(cleaned), cleaned

def backoff_sleep(k): time.sleep(min(20, 1.8**k + random.random()))

# ---- batching parameters (start with a small limit to confirm) ----
ROW_START   = 0
ROW_LIMIT   = None          # ← first test; set to None after you see progress
TEMP        = 0.2
MAX_TOKENS  = 900
SAVE_EVERY  = 2
MAX_ATTEMPTS_PER_ROW = 3

# ---- resume: skip rows already saved ----
batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
processed = set()
if batch_csv_path.exists():
    try:
        prior = pd.read_csv(batch_csv_path, usecols=["source_row_index"])
        processed = set(prior["source_row_index"].astype(int).tolist())
        print(f"Resume mode: {len(processed)} rows already done; will skip them.")
    except Exception as e:
        print("Resume disabled (could not read prior batch CSV):", e)

# ---- plan iteration ----
end_idx = len(df) if ROW_LIMIT is None else min(len(df), ROW_START + ROW_LIMIT)
indexes = [i for i in range(ROW_START, end_idx) if i not in processed]
print(f"Planned rows to process: {len(indexes)} of {len(df)} (from {ROW_START} to {end_idx-1})")
if not indexes:
    print("Nothing to do: either ROW_LIMIT=0, or all planned rows already in batch CSV,")
    print("or ROW_START >= end_idx. If you want a clean re-run, delete previous batch files:")
    print(" (Path(OUTPUTS_DIR)/'Job_Classifications_Batch.csv').unlink(missing_ok=True)")
    print(" (Path(OUTPUTS_DIR)/'Batch_Errors.json').unlink(missing_ok=True)")

records, errors = [], []
start_time = time.time()

# ---- loop ----
for k, i in enumerate(indexes, start=1):
    r = df.iloc[i]
    text = full_text_for_row(r)

    t0 = time.time()
    parsed = None
    raw    = ""

    for attempt in range(MAX_ATTEMPTS_PER_ROW):
        try:
            parsed, raw = call_model_with_text(text, TEMP, MAX_TOKENS)
            break
        except Exception as e:
            msg = str(e)
            if attempt == MAX_ATTEMPTS_PER_ROW - 1:
                snippet = (coerce_to_json_str(raw) if raw else "")[:600]
                errors.append((i, "exception", msg[:500], snippet))
            backoff_sleep(attempt)

    if parsed:
        try:
            for rec in parsed.job_classification_table:
                row_out = rec.model_dump()
                row_out["source_row_index"] = i
                row_out["model_used"] = MODEL_ID
                records.append(row_out)
        except Exception as e:
            errors.append((i, "parse_collect_error", str(e)[:300], (raw or "")[:300]))
    else:
        cleaned = coerce_to_json_str(raw) if raw else ""
        errors.append((i, "no_parsed_output", cleaned[:600]))

    # checkpoint save
    if (k % SAVE_EVERY == 0) or (k == len(indexes)):
        if records:
            if batch_csv_path.exists():
                try:
                    prev = pd.read_csv(batch_csv_path)
                    merged = pd.concat([prev, pd.DataFrame(records)], ignore_index=True)
                    merged.drop_duplicates(subset=["source_row_index","job_title_original","new_job_title"], inplace=True)
                    merged.to_csv(batch_csv_path, index=False, encoding="utf-8")
                except Exception:
                    pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            else:
                pd.DataFrame(records).to_csv(batch_csv_path, index=False, encoding="utf-8")
            print(f"Checkpoint: wrote {len(pd.read_csv(batch_csv_path))} rows to batch CSV.")
            records = []
        Path(OUTPUTS_DIR, "Batch_Errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")

    print(f"[{k}/{len(indexes)}] row {i} in {time.time()-t0:.1f}s | total {(time.time()-start_time)/60:.1f} min | "
          f"ok so far {k - len(errors)} | err {len(errors)}")

# copy batch → single so housekeeping/master sees it
if batch_csv_path.exists():
    dst = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
    shutil.copy2(batch_csv_path, dst)
    print("📄 Copied batch to:", dst)

print("✅ Batch complete. Files in:", OUTPUTS_DIR)


=== Batch v3.1 start ===
MODEL_ID: gpt-4o-2024-11-20
Rows in df: 35
OUTPUTS_DIR: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs
openai SDK: 1.109.1
✅ Read Ground Truth Masterfile.csv with encoding=cp1252
Context block chars: 7734
supports_parse_schema: False | supports_create_schema: False
Planned rows to process: 35 of 35 (from 0 to 34)
[1/35] row 0 in 5.6s | total 0.1 min | ok so far 1 | err 0
Checkpoint: wrote 2 rows to batch CSV.
[2/35] row 1 in 5.0s | total 0.2 min | ok so far 2 | err 0
[3/35] row 2 in 5.2s | total 0.3 min | ok so far 3 | err 0
Checkpoint: wrote 4 rows to batch CSV.
[4/35] row 3 in 5.2s | total 0.4 min | ok so far 4 | err 0
[5/35] row 4 in 4.0s | total 0.4 min | ok so far 5 | err 0
Checkpoint: wrote 6 rows to batch CSV.
[6/35] row 5 in 7.6s | total 0.5 min | ok so far 6 | err 0
[7/35] row 6 in 4.8s | total 0.6 min | ok so far 7 | err 0
Checkpoint: wrote 8 rows to batch CSV.
[8/35] row 7 in 5.3s | total 0.7 min | ok so far 8 | err 0

In [29]:
# ===== Cell 16.54 — Live peek while batch runs =====
from pathlib import Path
import pandas as pd

p = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if p.exists():
    dfb = pd.read_csv(p)
    print("Rows saved so far:", len(dfb))
    # show last few and a quick look at which source rows are pending
    display(dfb.tail(5))
    if "source_row_index" in dfb.columns and 'df' in globals():
        done = set(dfb["source_row_index"].astype(int))
        pending = [i for i in range(len(df)) if i not in done]
        print("Remaining rows:", len(pending), "| next up:", pending[:10])
else:
    print("No batch file yet at:", p)


Rows saved so far: 35


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification,source_row_index,model_used
30,Dir Technology Strategy,Technology Strategy Director,Director,I,The job title 'Dir Technology Strategy' indica...,30,gpt-4o-2024-11-20
31,Teacher Grade 8 Social Studies,Social Studies Teacher Grade 8,Teacher,NaN,The role focuses on instructional delivery and...,31,gpt-4o-2024-11-20
32,Principal Asst ES,Principal Assistant I,Assistant,I,The job title 'Principal Asst ES' suggests a s...,32,gpt-4o-2024-11-20
33,Tech Maintenance and Fencing,Maintenance Technician I,Technician,I,The job title 'Tech Maintenance and Fencing' s...,33,gpt-4o-2024-11-20
34,Teacher Music Band,Music Band Teacher I,Teacher,I,The role 'Teacher Music Band' aligns with the ...,34,gpt-4o-2024-11-20


Remaining rows: 0 | next up: []


# **Version 6.0**
> Batch audit: counts, parameters, error preview   
> Sanity Check  

In [30]:
# ===== Cell 16.55 — Batch audit: counts, parameters, error preview =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run Cell 4 first (creates OUTPUTS_DIR)."
assert 'df' in globals(), "Run Cell 4 first (loads df)."

print("Total rows in input df:", len(df))

batch_csv_path = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch_csv_path.exists():
    dfb = pd.read_csv(batch_csv_path)
    print("Rows saved in batch CSV:", len(dfb))
    if "source_row_index" in dfb.columns:
        done = sorted(dfb["source_row_index"].astype(int).unique().tolist())
        print("First 10 processed row indexes:", done[:10])
        print("Last 10 processed row indexes:", done[-10:])
    else:
        print("Note: 'source_row_index' column missing in batch CSV.")
else:
    print("⚠️ No batch CSV found at:", batch_csv_path)

errors_path = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errors_path.exists():
    try:
        errs = json.loads(errors_path.read_text())
        print("Error entries:", len(errs))
        for j, e in enumerate(errs[:5]):
            print(f"  {j+1}.", e if isinstance(e, str) else (e[0:2] if isinstance(e, list) else e))
    except Exception as e:
        print("Could not read Batch_Errors.json:", e)
else:
    print("No Batch_Errors.json present — either none failed or nothing ran.")


Total rows in input df: 35
Rows saved in batch CSV: 35
First 10 processed row indexes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last 10 processed row indexes: [25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
Error entries: 0


In [31]:
# ===== Cell 16.56 — Canonicalize predictions; enforce closed set & level rules =====
import glob, re, json
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
VALID_ROLES   = globals().get("VALID_ROLES", [])

def _newest(globs):
    hits=[]; [hits.extend(glob.glob(str(OUTPUTS_DIR / g))) for g in globs]
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return hits[0] if hits else None

pred_path = _newest(["Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
if not pred_path: raise FileNotFoundError("No predictions found under outputs/. Run 16.5 first.")
print("[16.56] Canonicalizing:", pred_path)
df = pd.read_csv(pred_path, engine="python")

def _cols(x): return {c.lower(): c for c in x.columns}
pc = _cols(df)
major = pc.get("major_role_group") or pc.get("major group") or pc.get("major")
minor = pc.get("minor_sub_group")  or pc.get("minor") or pc.get("level")
title = pc.get("job description name") or pc.get("original_job_title") or pc.get("original job title")
idcol = pc.get("row_id") or pc.get("source_row_index")

def canonical_role(s: str) -> str:
    if not isinstance(s, str): s = str(s or "")
    raw = s.strip(); low = raw.lower()
    for k,v in ROLE_SYNONYMS.items():
        if k in low: return v
    for r in VALID_ROLES:
        if low == r.lower(): return r
    for r in sorted(VALID_ROLES, key=lambda x: -len(x)):
        if r.lower() in low or low in r.lower(): return r
    return raw.strip().title()

def normalize_minor(val: str) -> str:
    s = (str(val) if val is not None else "").strip().upper()
    if s in {"LEAD","I","II","III"}: return "Lead" if s=="LEAD" else s
    if s in {"IV","4","V","5","VI","6"}: return "Lead"  # IV/V/VI → Lead (policy)
    m = re.search(r"\b([1-6]|I{1,6}|V)\b", s)
    if m:
        return {"1":"I","2":"II","3":"III","4":"Lead","5":"Lead","6":"Lead","V":"Lead"}.get(m.group(1),"I")
    # heuristics: infer Lead from title
    if title and df[title].astype(str).str.contains(r"\blead\b", case=False, regex=True).any():
        return "Lead"
    return "I"

# Title-aware fallbacks
def _title_hint_role(row):
    t = str(row.get(title,"") if title else "").lower()
    if "teacher" in t: return "Teacher"
    if "principal asst" in t or "asst principal" in t or "assistant principal" in t: return "Assistant Principal"
    if "principal" in t: return "Principal"
    if "restorative" in t or "transition" in t: return "Coach"
    return None

# Apply canonical roles; rescue out-of-set via title hints
fixed = []
policy_notes = []
for i, r in df.iterrows():
    chosen = canonical_role(r[major]) if major else str(r.get(major,""))
    if chosen not in VALID_ROLES:
        hint = _title_hint_role(r)
        if hint and hint in VALID_ROLES:
            fixed.append(hint)
            policy_notes.append((i, f"Role '{r[major]}' → '{hint}' (title hint)"))
        else:
            # last resort: if 'teacher' present → Teacher, else Specialist
            fallback = "Teacher" if re.search(r"\bteacher\b", str(r), flags=re.I) else "Specialist"
            fixed.append(fallback)
            policy_notes.append((i, f"Role '{r[major]}' → '{fallback}' (fallback)"))
    else:
        fixed.append(chosen)
df[major] = fixed

if minor:
    df[minor] = df[minor].apply(normalize_minor)

out_path = str(Path(pred_path).with_name(Path(pred_path).stem + "_canon.csv"))
df.to_csv(out_path, index=False, encoding="utf-8")
print("[16.56] Wrote canonicalized predictions ->", out_path)
if policy_notes:
    print("[16.56] Policy fixes (sample):")
    for i,msg in policy_notes[:10]: print(f"  row {i}: {msg}")
    more = len(policy_notes)-10
    if more>0: print(f"  (+{more} more)")


[16.56] Canonicalizing: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch.csv
[16.56] Wrote canonicalized predictions -> /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch_canon.csv
[16.56] Policy fixes (sample):
  row 5: Role 'Supervisor' → 'Specialist' (fallback)


In [32]:
# ===== Cell 16.561 — Translator canonicalization hotfix =====
import re, glob
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))

def _latest(globs):
    hits=[]
    for g in globs: hits += glob.glob(str(OUTPUTS_DIR / g))
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return hits[0] if hits else None

# Prefer the already-canonicalized file from 16.56
pred_path = _latest(["*canon.csv"]) or _latest(["Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
if not pred_path:
    raise FileNotFoundError("No predictions found; run 16.5 then 16.56 first.")
print("[16.561] Fixing Translator rows in:", pred_path)

df = pd.read_csv(pred_path, engine="python")
lc = {c.lower(): c for c in df.columns}
major = lc.get("major_role_group") or lc.get("major") or lc.get("major group")
minor = lc.get("minor_sub_group")  or lc.get("minor") or lc.get("level")
title = lc.get("job description name") or lc.get("original_job_title") or lc.get("original job title")
ksacs = lc.get("knowledge, skills and abilities")

def _is_translator_like(row):
    t = " ".join([
        str(row.get(title,"") if title else ""),
        str(row.get(ksacs,"") if ksacs else ""),
    ]).lower()
    return bool(re.search(r"\btranslator|interpreter\b", t)) or \
           any(k in t for k in ["language access","bilingual","multilingual","parent outreach"])

def _norm_level(s):
    s = (str(s) if s is not None else "").strip().upper()
    if s in {"LEAD","I","II","III"}: return "Lead" if s=="LEAD" else s
    if s in {"IV","4","V","5","VI","6"}: return "Lead"
    m = re.search(r"\b([1-6]|I{1,6}|V)\b", s)
    if m: return {"1":"I","2":"II","3":"III","4":"Lead","5":"Lead","6":"Lead","V":"Lead"}[m.group(1)]
    # hint from title level token
    t = str(row.get(title,"") if title else "")
    if re.search(r"\bI{1}\b", t): return "I"
    if re.search(r"\bII{1}\b", t): return "II"
    if re.search(r"\bIII{1}\b", t): return "III"
    if re.search(r"\blead\b", t, flags=re.I): return "Lead"
    return "I"

fix_ct = 0
for i, row in df.iterrows():
    if _is_translator_like(row) and major:
        if str(row[major]).strip().lower() not in {"translator"}:
            df.at[i, major] = "Translator"
            fix_ct += 1
    if minor:
        df.at[i, minor] = _norm_level(row[minor])

out_path = str(Path(pred_path).with_name(Path(pred_path).stem + "_canon2.csv"))
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"[16.561] Applied {fix_ct} Translator fixes. Wrote -> {out_path}")


[16.561] Fixing Translator rows in: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch_canon.csv
[16.561] Applied 0 Translator fixes. Wrote -> /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch_canon_canon2.csv


In [ ]:
# ===== Cell 16.565 — Deterministic overrides (robust cols, misspellings/abbrs, Lead enforcement) =====
import re, glob
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))
ROLE_SYNONYMS = globals().get("ROLE_SYNONYMS", {})
VALID_ROLES   = set(globals().get("VALID_ROLES", []))

# Roles we will allow even if VALID_ROLES is missing/misaligned (safety valve)
ALLOWED_FORCE = {
    "Teacher","Coach","Principal","Assistant Principal","Supervisor",
    "Executive Director","Translator","Director","Specialist","Manager"
}

def _latest(globs):
    hits=[]
    for g in globs: hits += glob.glob(str(OUTPUTS_DIR / g))
    hits = sorted(hits, key=lambda p: Path(p).stat().st_mtime, reverse=True)
    return Path(hits[0]) if hits else None

pred_path = _latest(["*canon3.csv","*canon2.csv","*canon.csv",
                     "Job_Classifications_Batch*.csv","classified_job_descriptions*.csv"])
if not pred_path:
    raise FileNotFoundError("No predictions found; run 16.5 and 16.56 first.")
print("[16.565] Reading:", pred_path)

df = pd.read_csv(pred_path, engine="python")

def _norm(s): return re.sub(r"[^a-z0-9]+","", str(s).lower())

def _get_col(df, *cands):
    want = {_norm(c) for c in cands}
    for c in df.columns:
        if _norm(c) in want: return c
    for c in df.columns:
        n = _norm(c)
        if any(t in n for t in want): return c
    return None

major = _get_col(df, "major_role_group","major role group","major")
minor = _get_col(df, "minor_sub_group","minor sub-group","minor subgroup","minor","level")
title = _get_col(df, "job description name","original_job_title","original job title","job title","title")
ksacs = _get_col(df, "knowledge, skills and abilities","ksacs","knowledge skills and abilities")
efunc = _get_col(df, "essential functions")
psumm = _get_col(df, "position summary")

print("[16.565] Columns:",
      "\n   major =", major, "\n   minor =", minor, "\n   title =", title,
      "\n   ksacs =", ksacs, "\n   essential =", efunc, "\n   summary =", psumm)

if not major: raise ValueError("[16.565] Could not resolve the major role column.")

def _s(x):
    try: return "" if pd.isna(x) else str(x).strip()
    except Exception: return (str(x) if x is not None else "").strip()

def _has(text, pats):
    t = _s(text).lower()
    return any(re.search(p, t) for p in pats)

# --- Patterns including abbreviations/misspellings ---
P_TEACHER     = [r"\bteacher\b", r"\binstructor\b", r"\binstr\.?\b"]
P_COACH       = [r"\bcoach\b", r"instructional coach", r"\bplc\b", r"professional development",
                 r"coaching cycles", r"restorative practice", r"restorative practices", r"\btransition\b", r"\bjdc\b"]
P_AP          = [r"\bassistant principal\b", r"\bass?tant principal\b", r"\basst principal\b",
                 r"\bassistant\-principal\b", r"\ba\.?p\.?\b(?!\w)"]
P_PRINCIPAL   = [r"\bprincipal\b(?!\s*assistant)"]
P_SUPERVISOR  = [r"\bsupervisor\b"]
P_MGR_CREW    = [r"\bmgr\b.*(tech|crew|field|maintenance|repair)",
                 r"\bmanager\b.*(tech|crew|field|maintenance|repair)",
                 r"\b(technicians?|crew|maintenance|repair)\b"]
P_EXEC_DIR    = [r"\bexecutive director\b", r"\bexec\.?\s*director\b", r"\bexecutive dir\b", r"\bexec dir\b"]
P_TRANSLATOR  = [r"\btranslator\b", r"\binterpreter\b", r"language access", r"parent outreach.*translator"]

P_LEAD        = [r"\blead\b"]
P_BAD_LEVEL   = [r"\bIV\b|\b4\b|\bV\b|\b5\b|\bVI\b|\b6\b"]
P_ROMAN_I     = [r"\bI\b(?![A-Z])"]; P_ROMAN_II = [r"\bII\b(?![A-Z])"]; P_ROMAN_III = [r"\bIII\b(?![A-Z])"]
P_ARABIC_1    = [r"\b1\b"]; P_ARABIC_2 = [r"\b2\b"]; P_ARABIC_3 = [r"\b3\b"]

def _blob(row):
    parts = [title, psumm, efunc, ksacs]
    return " ".join(_s(row.get(c)) for c in parts if c).lower()

def _level_from_title(row):
    t = _s(row.get(title)) if title else ""
    if _has(t, P_LEAD): return "Lead"
    if _has(t, P_ROMAN_III) or _has(t, P_ARABIC_3): return "III"
    if _has(t, P_ROMAN_II)  or _has(t, P_ARABIC_2): return "II"
    if _has(t, P_ROMAN_I)   or _has(t, P_ARABIC_1): return "I"
    return None

def _norm_minor(val, row):
    s = _s(val).upper()
    if s in {"LEAD","I","II","III"}: return "Lead" if s=="LEAD" else s
    if _has(s, P_BAD_LEVEL) or (title and _has(row.get(title,""), P_BAD_LEVEL)): return "Lead"
    hint = _level_from_title(row)
    return hint or "I"

def _force_role(target, cur_major):
    # prefer VALID_ROLES; else allow ALLOWED_FORCE to break stalemates
    if target in VALID_ROLES: return target
    low = target.lower()
    for k,v in (ROLE_SYNONYMS or {}).items():
        if k in low and v in VALID_ROLES: return v
    return target if target in ALLOWED_FORCE else cur_major

fix_ct_major = 0
fix_ct_minor = 0
notes = []

for i, row in df.iterrows():
    cur_major = _s(row.get(major))
    t_title   = _s(row.get(title))
    t_blob    = _blob(row)

    target = None
    if _has(t_title, P_AP):                                   target = "Assistant Principal"
    elif _has(t_title, P_PRINCIPAL):                          target = "Principal"
    elif _has(t_title, P_EXEC_DIR):                           target = "Executive Director"
    elif _has(t_title, P_TRANSLATOR) or _has(t_blob, P_TRANSLATOR): target = "Translator"
    elif _has(t_title, P_COACH) or _has(t_blob, P_COACH):     target = "Coach"
    elif _has(t_title, P_SUPERVISOR) or _has(t_blob, P_MGR_CREW): target = "Supervisor"
    elif _has(t_title, P_TEACHER) or _has(t_blob, P_TEACHER): target = "Teacher"

    if target and target != cur_major:
        chosen = _force_role(target, cur_major)
        if chosen != cur_major:
            df.at[i, major] = chosen
            fix_ct_major += 1
            if len(notes) < 18:
                notes.append(f"row {i}: '{t_title[:60]}'  {cur_major} → {chosen}")

    if minor:
        new_m = _norm_minor(row.get(minor), row)
        if new_m != _s(row.get(minor)):
            df.at[i, minor] = new_m
            fix_ct_minor += 1

out_path = pred_path.with_name(pred_path.stem + "_canon4.csv")
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"[16.565] Major fixes: {fix_ct_major} | Level fixes: {fix_ct_minor}")
if notes:
    print("[16.565] Sample major-role fixes:")
    for msg in notes: print("  -", msg)
print("[16.565] Wrote ->", out_path)


In [ ]:
# ===== Cell 16.567 — Expected clarification overrides (from your Results vs Expected files) =====
import glob, re
from pathlib import Path
import pandas as pd

OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR","./outputs"))

def _latest(paths):
    hits=[]
    for p in paths: hits += glob.glob(p, recursive=True)
    hits = sorted(hits, key=lambda s: Path(s).stat().st_mtime, reverse=True)
    return Path(hits[0]) if hits else None

# Find the newest canon file from 16.565
pred_path = _latest([str(OUTPUTS_DIR / "*canon4.csv")]) or \
            _latest([str(OUTPUTS_DIR / "*canon3.csv"), str(OUTPUTS_DIR / "*canon2.csv"), str(OUTPUTS_DIR / "*canon.csv")])
if not pred_path: raise FileNotFoundError("No canon file from 16.565 found.")

# Find the newest “Results vs Expected” sheet (xlsx or csv)
rve = _latest([
    str(OUTPUTS_DIR.parent / "**/Results vs Expected*.xlsx"),
    str(OUTPUTS_DIR.parent / "**/Results vs Expected*.csv"),
    str(OUTPUTS_DIR / "Results vs Expected*.xlsx"),
    str(OUTPUTS_DIR / "Results vs Expected*.csv"),
    "**/Results vs Expected*.xlsx", "**/Results vs Expected*.csv"
])
if not rve:
    print("[16.567] No Results vs Expected file found — skipping this step.")
    display(pred_path)
else:
    print("[16.567] Reading predictions:", pred_path)
    print("[16.567] Reading Expected sheet:", rve)

    df = pd.read_csv(pred_path, engine="python")
    # read expected
    if rve.suffix.lower() == ".xlsx":
        exp = pd.read_excel(rve)
    else:
        exp = pd.read_csv(rve, engine="python")

    def _norm(s): return re.sub(r"\s+"," ", str(s).strip().lower()) if pd.notna(s) else ""
    # resolve columns
    def _pick(cols, *cands):
        low = {c.lower(): c for c in cols}
        for c in cands:
            if c.lower() in low: return low[c.lower()]
        # fuzzy
        for c in cols:
            n = _norm(c)
            if any(_norm(x) in n for x in cands): return c
        return None

    title_df = _pick(df.columns, "Job Description Name","Original Job Title","Original Job Title","Title")
    major_df = _pick(df.columns, "Major Role Group","major_role_group","Major")
    minor_df = _pick(df.columns, "Minor Sub-Group","minor_sub_group","Minor","Level")

    title_ex = _pick(exp.columns, "Job Description Name","Original Job Title","Title","Job Name")
    clar_ex  = _pick(exp.columns, "Expected clarification","Expected Clarification","Expected","Expectation","Notes")

    if not (title_df and title_ex and clar_ex):
        raise ValueError("[16.567] Could not resolve required columns (title / expected clarification).")

    # build expected map
    ROLE_LIST = ["Teacher","Coach","Principal","Assistant Principal","Supervisor",
                 "Executive Director","Translator","Director","Specialist","Manager","Advisor","Technician","Analyst"]
    LEVELS    = ["Lead","I","II","III"]
    def _parse_exp(txt):
        s = str(txt or "").strip()
        roles = [r for r in ROLE_LIST if re.search(rf"\b{re.escape(r)}\b", s, flags=re.I)]
        lvls  = [lv for lv in LEVELS if re.search(rf"\b{lv}\b", s, flags=re.I)]
        return (roles[0] if roles else None, lvls[0] if lvls else None)

    exp = exp[[title_ex, clar_ex]].dropna(how="all")
    exp["_k"] = exp[title_ex].map(_norm)
    exp["__target_role__"], exp["__target_level__"] = zip(*exp[clar_ex].map(_parse_exp))
    exp_map = exp.set_index("_k")[["__target_role__","__target_level__"]].to_dict(orient="index")

    # apply to df
    fix_major = fix_minor = 0
    df["_k"] = df[title_df].map(_norm)
    for i, row in df.iterrows():
        k = row["_k"]
        if k in exp_map:
            tr = exp_map[k]["__target_role__"]
            tl = exp_map[k]["__target_level__"]
            if tr:
                if tr != row.get(major_df):
                    df.at[i, major_df] = tr
                    fix_major += 1
            if tl and minor_df:
                if tl != row.get(minor_df):
                    df.at[i, minor_df] = tl
                    fix_minor += 1

    out_path = pred_path.with_name(pred_path.stem + "_canon5.csv")
    df.drop(columns=["_k"], errors="ignore").to_csv(out_path, index=False, encoding="utf-8")
    print(f"[16.567] Forced by Expected: major={fix_major} | level={fix_minor}  -> {out_path}")


In [34]:
# ===== Cell 16.6 — Quick sanity check for current run =====
from pathlib import Path
import pandas as pd, json

assert 'OUTPUTS_DIR' in globals(), "Run your unique-run cell first (defines OUTPUTS_DIR)."

batch = Path(OUTPUTS_DIR) / "Job_Classifications_Batch.csv"
if batch.exists():
    dfb = pd.read_csv(batch)
    print("✅ Batch rows in this run:", len(dfb))
    display(dfb.head(5))
else:
    print("⚠️ No batch file found at", batch)

errs = Path(OUTPUTS_DIR) / "Batch_Errors.json"
if errs.exists():
    e = json.loads(Path(errs).read_text())
    print("⚠️ Rows with errors:", len(e))
    if e:
        print("First error:", e[0])


✅ Batch rows in this run: 35


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification,source_row_index,model_used
0,Spec Transition,Transition Specialist I,Specialist,I,The job title 'Spec Transition' aligns with th...,0,gpt-4o-2024-11-20
1,Teacher CTE Criminal Justice,Criminal Justice Instructor I,Instructor,I,The role involves teaching Criminal Justice as...,1,gpt-4o-2024-11-20
2,Teacher Ex Ed Vision,Exceptional Education Vision Teacher I,Teacher,I,The role focuses on providing specialized educ...,2,gpt-4o-2024-11-20
3,Coach Advocacy Center,Advocacy Center Coach I,Coach,I,The job title 'Coach Advocacy Center' suggests...,3,gpt-4o-2024-11-20
4,Teacher Ex Ed Life Skills,Exceptional Education Life Skills Teacher I,Teacher,I,The role focuses on teaching life skills to st...,4,gpt-4o-2024-11-20


⚠️ Rows with errors: 0


In [35]:
# Cell 16.97 — Run Summary & Quality Report (v6.0d: skip empty confidence files)

import os, io, json, re, math, glob
import pandas as pd
from pathlib import Path
from datetime import datetime

# ---------- robust readers ----------
def _read_table_robust(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")
    # Excel?
    if p.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    size = p.stat().st_size
    if size == 0:
        raise ValueError(f"File is empty (0 bytes): {path}")

    encodings = ["utf-8","utf-8-sig","cp1252","latin1","windows-1252","utf-16","utf-16le","utf-16be"]
    seps = [None, ",", "\t", ";", "|"]  # None = sniff
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] >= 1:
                    return df
            except Exception:
                continue

    # manual decode + sniff
    try:
        raw = p.read_bytes()
    except Exception:
        raw = b""
    for enc in encodings:
        try:
            txt = raw.decode(enc, errors="ignore")
            sample = txt.splitlines()[:10]
            scores = {",":0, "\t":0, ";":0, "|":0}
            for line in sample:
                for k in scores: scores[k] += line.count(k)
            guess = max(scores, key=scores.get)
            df = pd.read_csv(io.StringIO(txt), sep=guess, engine="python")
            if df.shape[1] >= 1:
                return df
        except Exception:
            continue

    # last resort
    try:
        return pd.read_excel(path)
    except Exception:
        pass

    raise ValueError(f"Unable to parse table from file: {path}")

def _safe_load_table(p: Path) -> pd.DataFrame:
    """Return empty DF if file is missing/empty/unparseable (and log)."""
    try:
        if not p.exists():
            print(f"[Report] Skipping (missing): {p}")
            return pd.DataFrame()
        if p.stat().st_size < 10:  # treat tiny files as empty
            print(f"[Report] Skipping (empty/tiny): {p} (size={p.stat().st_size} bytes)")
            return pd.DataFrame()
        return _read_table_robust(str(p))
    except Exception as e:
        print(f"[Report] Skipping (parse error): {p} — {e}")
        return pd.DataFrame()

def _cols(df): return {c.lower(): c for c in df.columns}

def _parse_expected_roles_levels(txt: str):
    if not isinstance(txt, str): txt = str(txt or "")
    ROLES = ["Director","Manager","Supervisor","Specialist","Analyst","Technician","Advisor",
             "Teacher","Coach","Liaison","Architect","Designer","Principal","Executive Director",
             "Lead Tech","Coordinator","Accountant","Partner"]
    LEVELS = ["Lead","I","II","III"]
    roles  = [r for r in ROLES if re.search(rf"\b{re.escape(r)}\b", txt, flags=re.I)]
    levels = [lv for lv in LEVELS if re.search(rf"\b{lv}\b", txt, flags=re.I)]
    return roles, levels

def _safe_mean(vals):
    vals = [v for v in vals if isinstance(v, (int,float)) and not math.isnan(v)]
    return sum(vals)/len(vals) if vals else float("nan")

def _newest_match(roots, patterns):
    hits = []
    for root in roots:
        try:
            for pat in patterns:
                hits.extend(glob.glob(str(Path(root) / "**" / pat), recursive=True))
        except Exception:
            pass
    if not hits: return None
    hits = sorted(hits, key=lambda p: (Path(p).stat().st_mtime), reverse=True)
    return hits[0]

# ---------- locate predictions ----------
OUTPUTS_DIR = Path(globals().get("OUTPUTS_DIR", "./outputs"))
pred_patterns = [
    "Job_Classifications_Batch*.csv",
    "Job_Classifications_SINGLE*.csv",
    "classified_job_descriptions_refined*.csv",
    "classified_job_descriptions*.csv",
    "Job_Classifications_Batch*.xlsx",
    "classified_job_descriptions*.xlsx",
]
search_roots = [
    OUTPUTS_DIR,
    OUTPUTS_DIR.parent,
    Path.cwd(),
    Path("/content"),
    Path("/content/drive/My Drive/Colab Notebooks/Run Results"),
]

pred_path = _newest_match(search_roots, pred_patterns)
if not pred_path:
    raise FileNotFoundError("No predictions file found. Run Cell 16 (and optionally 16.8) first.")
print(f"[Report] Using predictions: {pred_path}")

# co-locate companion files with predictions
OUTPUTS_DIR = Path(pred_path).parent
conf_json = OUTPUTS_DIR / "role_confidence_top5.json"
conf_top5 = OUTPUTS_DIR / "role_confidence_top5.csv"
conf_long = OUTPUTS_DIR / "role_confidence_indicators.csv"

# ---------- load tables (robust/forgiving) ----------
pred_df = _safe_load_table(Path(pred_path))
conf_map = {}
if conf_json.exists():
    try:
        conf_map = json.loads(conf_json.read_text(encoding="utf-8"))
    except Exception:
        conf_map = {}

conf_df  = _safe_load_table(conf_top5)
long_df  = _safe_load_table(conf_long)

# ---------- column resolution ----------
pc = _cols(pred_df)
major_col = pc.get("major_role_group") or pc.get("major group") or pc.get("major")
minor_col = pc.get("minor_sub_group") or pc.get("minor") or pc.get("level")
title_col = pc.get("job description name") or pc.get("original_job_title") or pc.get("original job title")
new_title_col = pc.get("new_job_title") or pc.get("new job title")
just_col  = pc.get("grouping_justification") or pc.get("justification") or pc.get("rationale")
exp_col   = pc.get("expected")  # optional

if not major_col or not minor_col:
    raise ValueError(
        "Predictions must include 'major_role_group' and 'minor_sub_group' (or detectable equivalents).\n"
        f"Columns seen: {list(pred_df.columns)}"
    )

# ---------- hit@k vs confidence shortlist ----------
def _hit_k(row_id, chosen_role, k=1):
    lst = conf_map.get(str(row_id)) or conf_map.get(int(row_id)) or []
    top = [d.get("role","") for d in lst[:k]]
    return any(re.fullmatch(rf"{re.escape(chosen_role)}", r, flags=re.I) for r in top)

def _sel_conf(row_id, chosen_role):
    lst = conf_map.get(str(row_id)) or conf_map.get(int(row_id)) or []
    for d in lst:
        if re.fullmatch(rf"{re.escape(chosen_role)}", d.get("role",""), flags=re.I):
            try:
                return float(d.get("confidence", float("nan")))
            except Exception:
                return float("nan")
    return float("nan")

work = pred_df.copy()
if "row_id" not in work.columns:
    work = work.reset_index().rename(columns={"index":"row_id"})

work["_hit1"] = work.apply(lambda r: _hit_k(int(r["row_id"]), str(r[major_col]), 1), axis=1) if conf_map else False
work["_hit3"] = work.apply(lambda r: _hit_k(int(r["row_id"]), str(r[major_col]), 3), axis=1) if conf_map else False
work["_sel_conf"] = work.apply(lambda r: _sel_conf(int(r["row_id"]), str(r[major_col])), axis=1) if conf_map else float("nan")

# ---------- optional eval vs Expected ----------
pass_rate = None
role_miss, lvl_miss = [], []
if exp_col:
    tmp = work.copy()
    tmp["_exp_blank"] = tmp[exp_col].astype(str).str.strip().eq("")
    def _pass_row(row):
        if row["_exp_blank"]:
            return True
        roles, levels = _parse_expected_roles_levels(str(row[exp_col]))
        ok_role = True
        ok_level = True
        if roles:
            ok_role = any(re.search(rf"\b{re.escape(r)}\b", str(row[major_col]), flags=re.I) for r in roles)
        if levels:
            ok_level = any(str(row[minor_col]).strip().upper() == lv.upper() for lv in levels)
        return ok_role and ok_level
    tmp["_pass"] = tmp.apply(_pass_row, axis=1)
    pass_rate = float(tmp["_pass"].mean())

    ROLES = ["Director","Manager","Supervisor","Specialist","Analyst","Technician","Advisor",
             "Teacher","Coach","Liaison","Architect","Designer","Principal","Executive Director",
             "Lead Tech","Coordinator","Accountant","Partner"]
    for rname in ROLES:
        want = tmp[exp_col].astype(str).str.contains(rf"\b{re.escape(rname)}\b", case=False, regex=True)
        if want.any():
            missed = int((~tmp[major_col].astype(str).str.contains(rf"\b{re.escape(rname)}\b", case=False, regex=True) & want).sum())
            role_miss.append((rname, int(want.sum()), missed))
    role_miss = sorted(role_miss, key=lambda x: (-x[2], x[0]))

    LEVELS = ["Lead","I","II","III"]
    for lv in LEVELS:
        want = tmp[exp_col].astype(str).str.contains(rf"\b{lv}\b", case=False, regex=True)
        if want.any():
            missed = int((tmp[minor_col].astype(str).str.upper() != lv.upper()) & want)
            lvl_miss.append((lv, int(want.sum()), missed))
    lvl_miss = sorted(lvl_miss, key=lambda x: (-x[2], x[0]))

# ---------- metrics ----------
def _safe_mean(vals):
    vals = [v for v in vals if isinstance(v, (int,float)) and not math.isnan(v)]
    return sum(vals)/len(vals) if vals else float("nan")

n = len(work)
hit1 = float(work["_hit1"].mean()) if conf_map else None
hit3 = float(work["_hit3"].mean()) if conf_map else None
avg_sel_conf = float(_safe_mean(work["_sel_conf"])) if conf_map else None

metrics = {
    "rows": n,
    "predictions_file": str(Path(pred_path)),
    "has_confidence_shortlist": bool(conf_map),
    "hit@1_chosen_in_shortlist": hit1,
    "hit@3_chosen_in_shortlist": hit3,
    "avg_confidence_of_chosen_role": avg_sel_conf,
    "expected_pass_rate_including_blanks": pass_rate,
}

# ---------- report ----------
lines = []
lines.append(f"# Run Quality Report — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC\n")
lines.append(f"- Predictions: `{pred_path}`")
lines.append(f"- Records: {n}")
if conf_map:
    lines.append(f"- Confidence shortlist: present for {len(conf_map)} records")
    lines.append(f"- hit@1 (chosen role == top1): {hit1:.3f}")
    lines.append(f"- hit@3 (chosen role ∈ top3):  {hit3:.3f}")
    if avg_sel_conf == avg_sel_conf:
        lines.append(f"- Avg confidence of chosen role: {avg_sel_conf:.3f}")
else:
    lines.append("- Confidence shortlist: not found (run Cell 15.2 earlier)")

if pass_rate is not None:
    lines.append(f"- Pass rate vs Expected (blank = pass): {pass_rate:.3f}")

if role_miss:
    lines.append("\n## Top role misses (expected_count, missed)")
    for r, cnt, miss in role_miss[:10]:
        lines.append(f"- {r:<12} expected={cnt:<3} missed={miss:<3}")
if lvl_miss:
    lines.append("\n## Level misses (expected_count, missed)")
    for lv, cnt, miss in lvl_miss:
        lines.append(f"- {lv:<5} expected={cnt:<3} missed={miss:<3}")

# disagreements sample
sample = pd.DataFrame()
if pass_rate is not None:
    tmp = work.copy()
    tmp["_exp_blank"] = tmp[exp_col].astype(str).str.strip().eq("")
    bad = tmp[(~tmp["_exp_blank"]) & (tmp[exp_col].astype(str).str.len()>0)].copy()
    def _is_fail(row):
        roles, levels = _parse_expected_roles_levels(str(row[exp_col]))
        ok_role  = True if not roles  else any(re.search(rf"\b{re.escape(r)}\b", str(row[major_col]), flags=re.I) for r in roles)
        ok_level = True if not levels else any(str(row[minor_col]).strip().upper()==lv.upper() for lv in levels)
        return not (ok_role and ok_level)
    bad["_fail"] = bad.apply(_is_fail, axis=1)
    sample = bad[bad["_fail"]].head(10)
elif conf_map:
    low = work.copy()
    low["_not_in_top3"] = ~low["_hit3"]
    low["_low_conf"] = low["_sel_conf"].apply(lambda v: (isinstance(v, (int,float))) and v < 0.35)
    sample = low[low[["_not_in_top3","_low_conf"]].any(axis=1)].head(10)

show_cols = [c for c in [title_col, exp_col, major_col, minor_col, new_title_col, just_col] if c]
sample_out = sample[show_cols].copy() if not sample.empty else pd.DataFrame(columns=show_cols)

# save
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
(report_path := OUTPUTS_DIR / "run_quality_report.md").write_text("\n".join(lines), encoding="utf-8")
pd.DataFrame([metrics]).to_csv(OUTPUTS_DIR / "hit_metrics.csv", index=False, encoding="utf-8")
sample_out.to_csv(OUTPUTS_DIR / "disagreements_sample.csv", index=False, encoding="utf-8")

print("[Report] Wrote:", report_path)
print("[Report] Wrote:", OUTPUTS_DIR / "hit_metrics.csv")
print("[Report] Wrote:", OUTPUTS_DIR / "disagreements_sample.csv")
print("\n=== SUMMARY ===")
print("\n".join(lines[:12]))
if not sample_out.empty:
    print("\n=== SAMPLE DISAGREEMENTS ===")
    display(sample_out)
else:
    print("\nNo disagreements to sample (either no Expected or all passed / were in top3 with decent confidence).")


[Report] Using predictions: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch_canon_canon2_canon4.csv
[Report] Wrote: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/run_quality_report.md
[Report] Wrote: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/hit_metrics.csv
[Report] Wrote: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/disagreements_sample.csv

=== SUMMARY ===
# Run Quality Report — 2025-09-24 20:21:47 UTC

- Predictions: `/content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Job_Classifications_Batch_canon_canon2_canon4.csv`
- Records: 35
- Confidence shortlist: present for 35 records
- hit@1 (chosen role == top1): 0.743
- hit@3 (chosen role ∈ top3):  0.800
- Avg confidence of chosen role: 0.945

=== SAMPLE DISAGREEMENTS ===


/tmp/ipython-input-468696567.py:240: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  lines.append(f"# Run Quality Report — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC\n")


,major_role_group,minor_sub_group,new_job_title,grouping_justification
0,Coach,I,Transition Specialist I,The job title 'Spec Transition' aligns with th...
5,Specialist,I,School Security Supervisor I,The job title 'Supv School Security' indicates...
10,Coach,I,Restorative Practice Specialist I,The job title 'Spec Restorative Practice (JDC)...
12,Specialist,II,Plumbing Specialist II,The job title 'Skilled Laborer Plumbing II' su...
19,Assistant,I,HR Information Assistant I,The job title 'Asst HR Information' suggests a...
32,Principal,I,Principal Assistant I,The job title 'Principal Asst ES' suggests a s...
33,Technician,I,Maintenance Technician I,The job title 'Tech Maintenance and Fencing' s...


In [36]:
#Cell 17
# Inspect parsed output (Responses API)
try:
    parsed  # from Cell 16
    print(parsed.model_dump_json(indent=2))
except NameError:
    print("No 'parsed' object found. Run Cell 16 first.")


{
  "job_classification_table": [
    {
      "job_title_original": "Teacher Music Band",
      "new_job_title": "Music Band Teacher I",
      "major_role_group": "Teacher",
      "minor_sub_group": "I",
      "grouping_justification": "The role 'Teacher Music Band' aligns with the 'Teacher' major role group due to its primary function of educating students in music and band-related subjects. The minor sub-group 'I' is assigned as it represents an entry-level teaching position, assuming no additional information about advanced responsibilities or seniority is provided. This classification is based on the functional alignment with teaching roles and the assumption of standard educational and certification requirements for teaching music in a school setting."
    }
  ],
  "narrative_rationale": "The job 'Teacher Music Band' was classified under the 'Teacher' major role group because it involves direct instruction and education of students in the subject of music and band. The minor sub-g

In [37]:
# Cell 17.5 — Build a response_dict from the Responses API parsed object
from pathlib import Path
import json
import pandas as pd

# Make sure Cell 16 ran (it defines `parsed`) and the run folders exist
assert 'parsed' in globals(), "Run Cell 16 first (it sets `parsed`)."
assert 'OUTPUTS_DIR' in globals(), "Run the unique-run cell first (defines OUTPUTS_DIR)."

# Convert the Pydantic objects to plain dicts
response_dict = {
    "job_classification_table": [rec.model_dump() for rec in parsed.job_classification_table],
    "narrative_rationale": parsed.narrative_rationale,
}

# Optional: preview the first rows
display(pd.DataFrame(response_dict["job_classification_table"]).head(10))

# Optional: save a pretty JSON alongside your other outputs
out_json = Path(OUTPUTS_DIR) / "Parsed_Response.json"
out_json.write_text(json.dumps(response_dict, indent=2), encoding="utf-8")
print("Saved:", out_json)

# Also return the dict so it shows below the cell
response_dict


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification
0,Teacher Music Band,Music Band Teacher I,Teacher,I,The role 'Teacher Music Band' aligns with the ...


Saved: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20250924_200625/outputs/Parsed_Response.json


{'job_classification_table': [{'job_title_original': 'Teacher Music Band',
   'new_job_title': 'Music Band Teacher I',
   'major_role_group': 'Teacher',
   'minor_sub_group': 'I',
   'grouping_justification': "The role 'Teacher Music Band' aligns with the 'Teacher' major role group due to its primary function of educating students in music and band-related subjects. The minor sub-group 'I' is assigned as it represents an entry-level teaching position, assuming no additional information about advanced responsibilities or seniority is provided. This classification is based on the functional alignment with teaching roles and the assumption of standard educational and certification requirements for teaching music in a school setting."}],
 'narrative_rationale': "The job 'Teacher Music Band' was classified under the 'Teacher' major role group because it involves direct instruction and education of students in the subject of music and band. The minor sub-group 'I' was chosen to reflect an en

In [38]:
#Cell 18
# Preview the saved classifications CSV (if present)
from pathlib import Path
import pandas as pd

csv_path = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
if csv_path.exists():
    display(pd.read_csv(csv_path).head(10))
else:
    print("No Job_Classifications.csv found in", OUTPUTS_DIR)


,job_title_original,new_job_title,major_role_group,minor_sub_group,grouping_justification,source_row_index,model_used
0,Spec Transition,Transition Specialist I,Specialist,I,The job title 'Spec Transition' aligns with th...,0,gpt-4o-2024-11-20
1,Teacher CTE Criminal Justice,Criminal Justice Instructor I,Instructor,I,The role involves teaching Criminal Justice as...,1,gpt-4o-2024-11-20
2,Teacher Ex Ed Vision,Exceptional Education Vision Teacher I,Teacher,I,The role focuses on providing specialized educ...,2,gpt-4o-2024-11-20
3,Coach Advocacy Center,Advocacy Center Coach I,Coach,I,The job title 'Coach Advocacy Center' suggests...,3,gpt-4o-2024-11-20
4,Teacher Ex Ed Life Skills,Exceptional Education Life Skills Teacher I,Teacher,I,The role focuses on teaching life skills to st...,4,gpt-4o-2024-11-20
5,Supv School Security,School Security Supervisor I,Supervisor,I,The job title 'Supv School Security' indicates...,5,gpt-4o-2024-11-20
6,Tech Grounds II,Grounds Technician II,Technician,II,The job title 'Tech Grounds II' suggests a tec...,6,gpt-4o-2024-11-20
7,Coord Gifted and Talented,Gifted and Talented Program Coordinator I,Coordinator,I,The role 'Coord Gifted and Talented' aligns wi...,7,gpt-4o-2024-11-20
8,Analyst ESEA Compliance,ESEA Compliance Analyst I,Analyst,I,The job title 'Analyst ESEA Compliance' sugges...,8,gpt-4o-2024-11-20
9,Teacher Social Studies Geography,Social Studies Teacher I,Teacher,I,The role focuses on instructional responsibili...,9,gpt-4o-2024-11-20


In [ ]:
# Cell 18.2 — Copy to Google Drive (includes Role Confidence tables)

from google.colab import drive
import os, shutil, datetime
from pathlib import Path

# Mount Drive
drive.mount('/content/drive')

# Base target in Drive
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Run Results'

# Unique run folder
timestamp = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
unique_folder_name = f'RUN_{timestamp}'
target_folder = os.path.join(base_target_folder, unique_folder_name)
os.makedirs(target_folder, exist_ok=True)

# Collect inputs/resources (best effort)
possible_inputs = [
    globals().get("BATCH_INPUT_CSV", "/content/Sample JDs.csv"),
    "/content/MNPS Roles.csv",
    "/content/MNPS KSACs.csv",
    "/content/Competency Extended Descriptions.csv",
    "/content/Korn_Ferry Lominger 38 Competencies.csv",
    globals().get("GROUND_TRUTH_CSV", "/content/Ground Truth Masterfile.csv"),
]

# Collect outputs
OUTPUTS_DIR = globals().get("OUTPUTS_DIR", "./outputs")
out_candidates = [
    os.path.join(OUTPUTS_DIR, "classified_job_descriptions.csv"),
    os.path.join(OUTPUTS_DIR, "classified_job_descriptions_refined.csv"),
    os.path.join(OUTPUTS_DIR, "classification_decision_log.csv"),
    os.path.join(OUTPUTS_DIR, "refinement_log.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_indicators.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_top5.csv"),
    os.path.join(OUTPUTS_DIR, "role_confidence_top5.json"),
]

files_to_copy = []
for p in possible_inputs + out_candidates:
    if p and os.path.exists(p):
        files_to_copy.append(p)

# Copy
for src in files_to_copy:
    try:
        fname = os.path.basename(src)
        dst = os.path.join(target_folder, fname)
        shutil.copy(src, dst)
        print(f"Copied: {fname}")
    except FileNotFoundError:
        print(f"Missing: {src}")
    except Exception as e:
        print(f"Error copying {src}: {e}")

print("\n[Drive] Copied files to:", target_folder)
